# DAZONE 2026 — Strong Uplift Modeling Notebook — TOP2 + Fold-safe Chi Filter

Bản này giữ lại **2 model tốt nhất** và bổ sung **uplift-aware feature selection** theo paper `Feature Selection Methods for Uplift Modeling and Heterogeneous Treatment Effect` (Zhao et al.).

Thay đổi chính:
- Dùng **Chi filter** để score feature theo mức độ tạo ra heterogeneous treatment effect.
- Feature selection chạy **bên trong từng fold**: chỉ dùng train fold để chọn feature, sau đó predict validation fold.
- Với mỗi fold, chọn `top-K` feature tốt nhất cho **cash** và **card**, sau đó lấy **union** để train/predict.
- Giữ nguyên pipeline chạy model phía sau, nhưng chỉ chạy **TOP2 model** để giảm chi phí.
- Đã giữ patch cache key theo `tr` và comment giải thích chủ đích fold-safe khi union cash/card.

## Danh sách model được giữ lại
| # | Model | Qini cash | Qini card | Qini tổng |
|---:|---|---:|---:|---:|
| 1 | **INT_RF** | +0.05424 | +0.03991 | **+0.09415** |
| 2 | **UPLIFT_RF_CIT** | +0.05721 | +0.03251 | **+0.08972** |


In [1]:
# ============================================================
# KAGGLE ENVIRONMENT SETUP
# ============================================================
# Trên Kaggle: bật Notebook Settings -> Internet -> On nếu muốn tự cài package còn thiếu.
# Nếu package đã có sẵn, cell này sẽ không cài lại.

RUN_INSTALL = True
INSTALL_OPTIONAL_CAUSAL_PACKAGES = True    # bật để cài causalml/utboost và chạy direct uplift tree/RF/UTBoost.
INSTALL_PYKAN = True                       # cần cho deep_kan.

import importlib.util
import subprocess
import sys


def is_installed(import_name: str) -> bool:
    return importlib.util.find_spec(import_name) is not None


def pip_install(package: str):
    print(f"Installing: {package}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


if RUN_INSTALL:
    required = [
        ("numpy", "numpy"),
        ("pandas", "pandas"),
        ("scipy", "scipy"),
        ("sklearn", "scikit-learn"),
        ("matplotlib", "matplotlib"),
        ("lightgbm", "lightgbm"),
        ("catboost", "catboost"),
        ("xgboost", "xgboost"),
        ("torch", "torch"),
        ("tqdm", "tqdm"),
        ("sympy", "sympy"),
        # pykan có thể gọi duecredit khi import. Thiếu duecredit thường chỉ là warning,
        # nhưng cài thêm để tránh dòng: Failed to import duecredit due to No module named 'duecredit'.
        ("duecredit", "duecredit"),
    ]
    if INSTALL_PYKAN:
        # pykan có thể cài từ PyPI. Nếu Kaggle lỗi version, dùng dòng GitHub bên dưới trong comment.
        # pip_install("git+https://github.com/KindXiaoming/pykan.git")
        required.append(("kan", "pykan"))
    if INSTALL_OPTIONAL_CAUSAL_PACKAGES:
        required += [("causalml", "causalml"), ("utboost", "utboost")]

    for import_name, package_name in required:
        if not is_installed(import_name):
            try:
                pip_install(package_name)
            except Exception as e:
                print(f"WARNING: Không cài được {package_name}: {repr(e)}")
                print("Notebook sẽ tự skip model phụ thuộc package này nếu package không import được.")
        else:
            print(f"OK: {import_name} already installed")
else:
    print("RUN_INSTALL=False -> bỏ qua cài package.")

# Nếu môi trường vẫn in warning duecredit từ package bên thứ ba, suppress nhẹ để output sạch hơn.
import warnings
warnings.filterwarnings("ignore", message=".*duecredit.*")


OK: numpy already installed
OK: pandas already installed
OK: scipy already installed
OK: sklearn already installed
OK: matplotlib already installed
OK: lightgbm already installed
OK: catboost already installed
OK: xgboost already installed
OK: torch already installed
OK: tqdm already installed
OK: sympy already installed
Installing: duecredit
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.5/91.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 2.2 MB/s eta 0:00:00
Installing: pykan
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 3.7 MB/s eta 0:00:00
Installing: causalml
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 3.9 MB/s eta 0:00:00
Installing: utboost
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 391.3/391.3 kB 13.9 MB/s eta 0:00:00


In [2]:
import gc
import json
import os
import sys
import time
import warnings
from pathlib import Path

# Tránh OpenMP/BLAS dùng quá nhiều thread khiến sklearn/torch chạy chậm trên một số runtime Kaggle.
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")

SEED = 42
N_SPLITS = 5
N_JOBS = 4

# ============================================================
# PRESET CHẠY MODEL
# ============================================================
# Chỉ cần sửa dòng dưới để chọn nhóm model muốn chạy:
#   "deepKAN"        : chỉ chạy Deep KAN thật từ pykan.
#   "KAN"            : chỉ chạy KAN ban đầu dạng spline/KAN-style estimator.
#   "MLP"            : chỉ chạy MLP.
#   "MLs"            : chạy tất cả model Machine Learning còn lại, loại trừ deep_kan, kan, mlp.
#   "requested_full" : chạy tất cả base models: deep_kan + kan + mlp + MLs.
#
# Alias được hỗ trợ: "deep_kan", "request_full", "request full", "request ful", "full".
RUN_PRESET = "requested_full"

RUN_PRESET_ALIASES = {
    "deepkan": "deepKAN",
    "deep_kan": "deepKAN",
    "deep-kan": "deepKAN",
    "pykan": "deepKAN",
    "kan": "KAN",
    "splinekan": "KAN",
    "spline_kan": "KAN",
    "mlp": "MLP",
    "mls": "MLs",
    "ml": "MLs",
    "machine_learning": "MLs",
    "machine learning": "MLs",
    "requested_full": "requested_full",
    "request_full": "requested_full",
    "request full": "requested_full",
    "request ful": "requested_full",
    "full": "requested_full",
}
RUN_PRESET = RUN_PRESET_ALIASES.get(str(RUN_PRESET).strip().lower(), RUN_PRESET)

# KAN ban đầu giữ nguyên: "kan" = SplineKANClassifier/SplineKANRegressor nhẹ theo kiểu spline additive.
# Deep KAN mới: "deep_kan" = dùng package pykan thật.
REQUESTED_BASE_MODELS = [
    "kan",          # KAN ban đầu, giữ nguyên: KAN-style spline estimator
    "deep_kan",     # Deep KAN thật từ pykan
    "mlp",          # Multi-layer Perceptron
    "svm",          # Linear SVM / Linear SVR
    "adaboost",     # AdaBoost
    "dt",           # Decision Tree
    "rf",           # Random Forest
    "extratrees",   # Extra Trees
    "lgbm",         # LightGBM
    "logistic",     # Logistic Regression / Ridge for continuous pseudo-outcome
    "knn",          # Distance KNN
    "fuzzy_knn",    # Fuzzy KNN
    "hgb",          # HistGradientBoosting
]

# Giữ lại model mạnh từ notebook cũ nếu package khả dụng.
EXTRA_STRONG_BASE_MODELS = ["cat", "xgb"]

# Nhóm model theo preset.
SPECIAL_NEURAL_MODELS = ["deep_kan", "kan", "mlp"]
ML_BASE_MODELS = [
    m for m in (REQUESTED_BASE_MODELS + EXTRA_STRONG_BASE_MODELS)
    if m not in SPECIAL_NEURAL_MODELS
]
ALL_BASE_MODELS = REQUESTED_BASE_MODELS + EXTRA_STRONG_BASE_MODELS

# Tất cả uplift/meta-learner đang được notebook hỗ trợ.
# Với preset deepKAN/KAN/MLP, ta chỉ giới hạn base model, KHÔNG giới hạn learner.
# Ví dụ RUN_PRESET="MLP" sẽ chạy T_MLP, S_MLP, INT_MLP, CT_MLP, TO_MLP, X_MLP, R_MLP, DR_MLP.
ALL_UPLIFT_LEARNERS = ["T", "S", "INT", "CT", "TO", "X", "R", "DR"]

# Hyperparameters chung theo từng preset.
if RUN_PRESET == "deepKAN":
    # 1) Chỉ chạy Deep KAN thật từ pykan.
    RUN_MODEL_KINDS = ["deep_kan"]
    RUN_LEARNERS = ALL_UPLIFT_LEARNERS
    LGB_ESTIMATORS = 120
    CAT_ITERATIONS = 80
    XGB_ESTIMATORS = 120
    TREE_ESTIMATORS = 80
    NN_MAX_ITER = 60
    UTB_ITERATIONS = 80
    DEEP_KAN_STEPS = 80
    DEEP_KAN_MAX_TRAIN_ROWS = 12000

elif RUN_PRESET == "KAN":
    # 2) Chỉ chạy KAN ban đầu dạng spline/KAN-style estimator.
    RUN_MODEL_KINDS = ["kan"]
    RUN_LEARNERS = ALL_UPLIFT_LEARNERS
    LGB_ESTIMATORS = 120
    CAT_ITERATIONS = 80
    XGB_ESTIMATORS = 120
    TREE_ESTIMATORS = 80
    NN_MAX_ITER = 60
    UTB_ITERATIONS = 80
    DEEP_KAN_STEPS = 40
    DEEP_KAN_MAX_TRAIN_ROWS = 6000

elif RUN_PRESET == "MLP":
    # 3) Chỉ chạy MLP.
    RUN_MODEL_KINDS = ["mlp"]
    RUN_LEARNERS = ALL_UPLIFT_LEARNERS
    LGB_ESTIMATORS = 120
    CAT_ITERATIONS = 80
    XGB_ESTIMATORS = 120
    TREE_ESTIMATORS = 80
    NN_MAX_ITER = 140
    UTB_ITERATIONS = 80
    DEEP_KAN_STEPS = 40
    DEEP_KAN_MAX_TRAIN_ROWS = 6000

elif RUN_PRESET == "MLs":
    # 4) Chạy tất cả Machine Learning còn lại, loại trừ deep_kan, kan, mlp.
    RUN_MODEL_KINDS = ML_BASE_MODELS
    RUN_LEARNERS = ALL_UPLIFT_LEARNERS
    LGB_ESTIMATORS = 520
    CAT_ITERATIONS = 360
    XGB_ESTIMATORS = 420
    TREE_ESTIMATORS = 320
    NN_MAX_ITER = 120
    UTB_ITERATIONS = 220
    DEEP_KAN_STEPS = 40
    DEEP_KAN_MAX_TRAIN_ROWS = 6000

elif RUN_PRESET == "requested_full":
    # 5) Chạy tất cả base models: deepKAN + spline KAN + MLP + MLs.
    RUN_MODEL_KINDS = ALL_BASE_MODELS
    RUN_LEARNERS = ALL_UPLIFT_LEARNERS
    LGB_ESTIMATORS = 520
    CAT_ITERATIONS = 360
    XGB_ESTIMATORS = 420
    TREE_ESTIMATORS = 320
    NN_MAX_ITER = 140
    UTB_ITERATIONS = 220
    DEEP_KAN_STEPS = 80
    DEEP_KAN_MAX_TRAIN_ROWS = 10000

else:
    raise ValueError(
        "RUN_PRESET không hợp lệ. Hãy chọn một trong: "
        "'deepKAN', 'KAN', 'MLP', 'MLs', 'requested_full'."
    )

# Optional direct uplift models.
# Nếu muốn chạy Uplift Tree/RF/IT/CIT/UTBoost thật, cần cài causalml/utboost ở cell install
# rồi đổi hai biến dưới thành True. Base preset ở trên không phụ thuộc hai biến này.
RUN_OPTIONAL_CAUSALML_TREE = True
RUN_OPTIONAL_UTBOOST = True

CAUSALML_TREE_CRITERIA = ["KL", "ED", "Chi", "CTS", "DDP", "IT", "CIT"]
CAUSALML_FOREST_CRITERIA = ["KL", "ED", "Chi", "CTS", "DDP", "IT", "CIT"]

# ============================================================
# SELECTIVE RUN — chỉ giữ lại TOP 2 model theo bảng Qini đã lọc Qini card âm
# ============================================================
# Mục tiêu: giảm chi phí chạy bằng cách chỉ train/evaluate đúng các model nằm trong top list.
# Các model khác được bỏ qua ở cell run.
TOP_MODEL_NAMES = [
    "INT_RF",
    "UPLIFT_RF_CIT",
]
SELECTED_MODEL_NAMES = set(TOP_MODEL_NAMES)

_META_LEARNER_KEYS = ["T", "S", "INT", "CT", "TO", "X", "R", "DR"]
_DIRECT_CRITERION_CANON = {"KL": "KL", "ED": "ED", "CHI": "Chi", "CTS": "CTS", "DDP": "DDP", "IT": "IT", "CIT": "CIT"}


def _is_selected_meta_model(name: str) -> bool:
    return any(name.startswith(prefix + "_") for prefix in _META_LEARNER_KEYS)


_selected_meta_models = [m for m in TOP_MODEL_NAMES if _is_selected_meta_model(m)]

# Chỉ giữ learner và base model xuất hiện trong top list.
RUN_LEARNERS = [lk for lk in _META_LEARNER_KEYS if any(m.startswith(lk + "_") for m in _selected_meta_models)]
RUN_MODEL_KINDS = []
for m in _selected_meta_models:
    learner_key, base_name = m.split("_", 1)
    base_kind = base_name.lower()
    if base_kind not in RUN_MODEL_KINDS:
        RUN_MODEL_KINDS.append(base_kind)

# Chỉ giữ direct uplift tree/RF và UTBoost xuất hiện trong top list.
def _criteria_from_selected(prefix: str):
    out = []
    for m in TOP_MODEL_NAMES:
        if m.startswith(prefix + "_"):
            raw = m.replace(prefix + "_", "", 1).upper()
            crit = _DIRECT_CRITERION_CANON.get(raw, raw)
            if crit not in out:
                out.append(crit)
    return out

CAUSALML_TREE_CRITERIA = _criteria_from_selected("UPLIFT_TREE")
CAUSALML_FOREST_CRITERIA = _criteria_from_selected("UPLIFT_RF")
RUN_OPTIONAL_CAUSALML_TREE = bool(CAUSALML_TREE_CRITERIA or CAUSALML_FOREST_CRITERIA)
RUN_OPTIONAL_UTBOOST = "UTBOOST" in SELECTED_MODEL_NAMES

print("Selective top model count:", len(TOP_MODEL_NAMES))
print("Selected meta models:", _selected_meta_models)
print("Selected direct tree criteria:", CAUSALML_TREE_CRITERIA)
print("Selected direct RF criteria:", CAUSALML_FOREST_CRITERIA)
print("Selected UTBoost:", RUN_OPTIONAL_UTBOOST)


# KNN rất nặng khi fit/predict trên full data, nên có subsampling để tránh treo runtime.
KNN_MAX_TRAIN_ROWS = 35000
KNN_NEIGHBORS = 75
FUZZY_KNN_POWER = 2.0

# Deep KAN pykan config. Deep KAN thường chậm hơn tree boosting rất nhiều.
DEEP_KAN_WIDTH_HIDDEN = [16, 8]
DEEP_KAN_GRID = 3
DEEP_KAN_K = 3
DEEP_KAN_BATCH_SIZE = 1024
DEEP_KAN_LR = 1e-3
DEEP_KAN_WEIGHT_DECAY = 1e-4
DEEP_KAN_USE_GPU = True

TOPK_ENSEMBLE = 8

print("Python:", sys.version.split()[0])
print("Preset:", RUN_PRESET)
print("N_SPLITS:", N_SPLITS, "| N_JOBS:", N_JOBS)
print("REQUESTED_BASE_MODELS:", REQUESTED_BASE_MODELS)
print("SPECIAL_NEURAL_MODELS:", SPECIAL_NEURAL_MODELS)
print("ML_BASE_MODELS:", ML_BASE_MODELS)
print("Run base models before availability filter:", RUN_MODEL_KINDS)
print("Learners:", RUN_LEARNERS)
print("Optional CausalML tree/RF:", RUN_OPTIONAL_CAUSALML_TREE)
print("Optional UTBoost:", RUN_OPTIONAL_UTBOOST)
print("CausalML tree criteria:", CAUSALML_TREE_CRITERIA)
print("CausalML forest criteria:", CAUSALML_FOREST_CRITERIA)
print("Deep KAN steps:", DEEP_KAN_STEPS, "| max rows:", DEEP_KAN_MAX_TRAIN_ROWS)


Selective top model count: 2
Selected meta models: ['INT_RF']
Selected direct tree criteria: []
Selected direct RF criteria: ['CIT']
Selected UTBoost: False
Python: 3.12.13
Preset: requested_full
N_SPLITS: 5 | N_JOBS: 4
REQUESTED_BASE_MODELS: ['kan', 'deep_kan', 'mlp', 'svm', 'adaboost', 'dt', 'rf', 'extratrees', 'lgbm', 'logistic', 'knn', 'fuzzy_knn', 'hgb']
SPECIAL_NEURAL_MODELS: ['deep_kan', 'kan', 'mlp']
ML_BASE_MODELS: ['svm', 'adaboost', 'dt', 'rf', 'extratrees', 'lgbm', 'logistic', 'knn', 'fuzzy_knn', 'hgb', 'cat', 'xgb']
Run base models before availability filter: ['rf']
Learners: ['INT']
Optional CausalML tree/RF: True
Optional UTBoost: False
CausalML tree criteria: []
CausalML forest criteria: ['CIT']
Deep KAN steps: 80 | max rows: 10000


In [3]:
AVAILABLE = {}
warnings.filterwarnings("ignore", message=".*duecredit.*")

# Core sklearn models.
try:
    import sklearn
    from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler, SplineTransformer
    from sklearn.linear_model import LogisticRegression, Ridge
    from sklearn.neural_network import MLPClassifier, MLPRegressor
    from sklearn.svm import LinearSVC, LinearSVR
    from sklearn.ensemble import (
        AdaBoostClassifier,
        AdaBoostRegressor,
        RandomForestClassifier,
        RandomForestRegressor,
        ExtraTreesClassifier,
        ExtraTreesRegressor,
        HistGradientBoostingClassifier,
        HistGradientBoostingRegressor,
    )
    from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
    from sklearn.neighbors import NearestNeighbors
    print("scikit-learn:", sklearn.__version__)
    for k in ["kan", "mlp", "svm", "adaboost", "dt", "rf", "extratrees", "logistic", "knn", "fuzzy_knn", "hgb"]:
        AVAILABLE[k] = True
except Exception as e:
    print("scikit-learn unavailable:", repr(e))
    for k in ["kan", "mlp", "svm", "adaboost", "dt", "rf", "extratrees", "logistic", "knn", "fuzzy_knn", "hgb"]:
        AVAILABLE[k] = False

try:
    import torch
    from kan import KAN as PyKAN
    AVAILABLE["deep_kan"] = True
    print("torch:", torch.__version__)
    print("pykan: available")
except Exception as e:
    AVAILABLE["deep_kan"] = False
    print("pykan/deep_kan unavailable:", repr(e))

try:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier, LGBMRegressor
    AVAILABLE["lgbm"] = True
    print("LightGBM:", lgb.__version__)
except Exception as e:
    AVAILABLE["lgbm"] = False
    print("LightGBM unavailable:", repr(e))

try:
    import catboost as cb
    from catboost import CatBoostClassifier, CatBoostRegressor
    AVAILABLE["cat"] = True
    print("CatBoost:", cb.__version__)
except Exception as e:
    AVAILABLE["cat"] = False
    print("CatBoost unavailable:", repr(e))

try:
    import xgboost as xgb
    from xgboost import XGBClassifier, XGBRegressor
    AVAILABLE["xgb"] = True
    print("XGBoost:", xgb.__version__)
except Exception as e:
    AVAILABLE["xgb"] = False
    print("XGBoost unavailable:", repr(e))

try:
    from causalml.inference.tree import UpliftTreeClassifier, UpliftRandomForestClassifier
    AVAILABLE["causalml"] = True
    print("causalml: available")
except Exception as e:
    AVAILABLE["causalml"] = False
    print("causalml unavailable; optional Uplift Tree/RF, Interaction Tree, CIT will be skipped.")

try:
    from utboost import UTBClassifier
    AVAILABLE["utboost"] = True
    print("UTBoost: available")
except Exception as e:
    AVAILABLE["utboost"] = False
    print("UTBoost unavailable; optional UTBoost will be skipped.")

RUN_MODEL_KINDS = [k for k in RUN_MODEL_KINDS if AVAILABLE.get(k, False)]
if RUN_PRESET == "deepKAN" and "deep_kan" not in RUN_MODEL_KINDS:
    raise ImportError(
        "RUN_PRESET='deepKAN' cần pykan. Trên Kaggle hãy bật Internet và chạy cell install: pip install pykan hoặc pip install git+https://github.com/KindXiaoming/pykan.git"
    )
if RUN_PRESET == "KAN" and "kan" not in RUN_MODEL_KINDS:
    raise ImportError("RUN_PRESET='KAN' cần scikit-learn để chạy SplineKANClassifier/SplineKANRegressor.")
if RUN_PRESET == "MLP" and "mlp" not in RUN_MODEL_KINDS:
    raise ImportError("RUN_PRESET='MLP' cần scikit-learn để chạy MLPClassifier/MLPRegressor.")
if not RUN_MODEL_KINDS:
    raise ImportError("Không có base learner nào khả dụng. Cần tối thiểu scikit-learn hoặc LightGBM/CatBoost/XGBoost/pykan.")

print("Run base models after availability filter:", RUN_MODEL_KINDS)
print("Total base models:", len(RUN_MODEL_KINDS))


scikit-learn: 1.6.1
torch: 2.10.0+cpu
pykan: available
LightGBM: 4.6.0
CatBoost: 1.2.10
XGBoost: 3.2.0
causalml: available
UTBoost: available
Run base models after availability filter: ['rf']
Total base models: 1


## 1. Load dữ liệu

In [4]:

def find_file_pair(feature_name="df_feature.csv", target_name="df_target.csv"):
    roots = []
    for p in [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd(), Path.cwd() / "data", Path("/mnt/data")]:
        if p.exists() and str(p.resolve()) not in ["/", "/mnt"]:
            roots.append(p)

    candidates = []
    for root in roots:
        direct_feat = root / feature_name
        direct_tgt = root / target_name
        if direct_feat.exists() and direct_tgt.exists():
            return direct_feat, direct_tgt
        try:
            for feat_path in root.rglob(feature_name):
                tgt_path = feat_path.parent / target_name
                if tgt_path.exists():
                    candidates.append((feat_path, tgt_path))
        except Exception:
            pass

    if candidates:
        candidates = sorted(candidates, key=lambda x: len(str(x[0])))
        return candidates[0]
    raise FileNotFoundError("Không tìm thấy df_feature.csv và df_target.csv")

feat_path, tgt_path = find_file_pair()
print("Feature path:", feat_path)
print("Target path :", tgt_path)

feat = pd.read_csv(feat_path)
tgt = pd.read_csv(tgt_path)
assert len(feat) == len(tgt), "df_feature và df_target phải cùng số dòng"

TARGET_CASH = "cash_sign_30d"
TARGET_CARD = "card_sign_30d"
TREAT_COL_BINARY = "treatment_cash"
TREAT_COL_RAW = "treatment"

assert TARGET_CASH in tgt.columns and TARGET_CARD in tgt.columns, "Thiếu target cash/card"

if TREAT_COL_BINARY in feat.columns:
    W = feat[TREAT_COL_BINARY].astype(int).to_numpy()  # 1 = Cash, 0 = Card
    TREAT_COL = TREAT_COL_BINARY
elif TREAT_COL_RAW in feat.columns:
    W = feat[TREAT_COL_RAW].astype(str).str.lower().map({"cash": 1, "card": 0}).astype(int).to_numpy()
    TREAT_COL = TREAT_COL_RAW
else:
    raise ValueError("Thiếu cột treatment_cash hoặc treatment")

Y_cash = tgt[TARGET_CASH].astype(int).to_numpy()
Y_card = tgt[TARGET_CARD].astype(int).to_numpy()

id_cols = [c for c in ["customer_id", "id", "Customer_ID"] if c in feat.columns]
ID_COL = id_cols[0] if id_cols else None

DROP_COLS = [TREAT_COL]
if ID_COL:
    DROP_COLS.append(ID_COL)
FEAT_COLS = [c for c in feat.columns if c not in DROP_COLS]

X_df = feat[FEAT_COLS].copy()
X = X_df.to_numpy(dtype=np.float32)

w_cash = (W == 1).astype(int)
w_card = (W == 0).astype(int)

print(f"N={len(X):,} | features={X.shape[1]}")
print(f"Treatment Cash: {W.sum():,} ({W.mean():.2%}) | Card: {(1-W).sum():,} ({(1-W).mean():.2%})")
print(f"Cash target rate: {Y_cash.mean():.4%} | Card target rate: {Y_card.mean():.4%}")
print("Feature NaN total:", int(pd.isna(X_df).sum().sum()))


# Giữ lại full feature matrix. Chi filter sẽ chọn subset riêng bên trong từng fold.
FULL_FEAT_COLS = list(FEAT_COLS)
X_full_df = X_df
X_full = X


Feature path: /kaggle/input/datasets/loinguyen57/dazone-data/df_feature.csv
Target path : /kaggle/input/datasets/loinguyen57/dazone-data/df_target.csv
N=136,992 | features=62
Treatment Cash: 75,612 (55.19%) | Card: 61,380 (44.81%)
Cash target rate: 5.9186% | Card target rate: 2.7111%
Feature NaN total: 0


In [5]:

def find_optional_test_feature():
    names = ["df_feature_test.csv", "test_feature.csv", "X_test.csv", "test.csv"]
    roots = [p for p in [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd(), Path.cwd() / "data", Path("/mnt/data")]
             if p.exists() and str(p.resolve()) not in ["/", "/mnt"]]
    for root in roots:
        for name in names:
            direct = root / name
            if direct.exists():
                return direct
            try:
                hits = list(root.rglob(name))
                if hits:
                    return sorted(hits, key=lambda p: len(str(p)))[0]
            except Exception:
                pass
    return None

test_path = find_optional_test_feature()
if test_path:
    test_feat = pd.read_csv(test_path)
    test_id = test_feat[ID_COL].copy() if ID_COL and ID_COL in test_feat.columns else pd.Series(np.arange(len(test_feat)), name="row_id")
    missing = [c for c in FEAT_COLS if c not in test_feat.columns]
    extra = [c for c in test_feat.columns if c not in FEAT_COLS + ([ID_COL] if ID_COL else [])]
    if missing:
        raise ValueError(f"Test thiếu feature: {missing[:10]}")
    X_test = test_feat[FEAT_COLS].to_numpy(dtype=np.float32)
    print("Test feature path:", test_path)
    print("Test N:", len(X_test), "| extra columns ignored:", extra[:10])
else:
    X_test = None
    test_id = None
    print("Không thấy test feature riêng. Notebook sẽ xuất OOF score trên train.")


# Full test matrix; fold-safe Chi filter sẽ slice đúng feature subset khi predict từng fold.
X_test_full = X_test


Không thấy test feature riêng. Notebook sẽ xuất OOF score trên train.


## 2. Metric Qini + fold cố định

In [6]:

_trapz = np.trapezoid if hasattr(np, "trapezoid") else np.trapz


def qini_curve(y, w_event, score):
    y = np.asarray(y, dtype=float)
    w_event = np.asarray(w_event, dtype=int)
    score = np.asarray(score, dtype=float)
    order = np.argsort(-score, kind="mergesort")
    y_s = y[order]
    w_s = w_event[order]
    n_t = max(int(w_event.sum()), 1)
    n_c = max(len(w_event) - n_t, 1)
    q = np.cumsum(y_s * w_s) / n_t - np.cumsum(y_s * (1 - w_s)) / n_c
    x = np.arange(1, len(y) + 1) / len(y)
    return np.r_[0.0, x], np.r_[0.0, q]


def oracle_score(y, w_event):
    y = np.asarray(y, dtype=int)
    w_event = np.asarray(w_event, dtype=int)
    return w_event * y - (1 - w_event) * y


def qini_metrics(y, w_event, score):
    x, q = qini_curve(y, w_event, score)
    auc_model = float(_trapz(q, x))
    ate = float(q[-1])
    auc_random = ate * 0.5
    _, q_perfect = qini_curve(y, w_event, oracle_score(y, w_event))
    auc_perfect = float(_trapz(q_perfect, x))
    qini_diff = auc_model - auc_random
    denom = auc_perfect - auc_random
    qini_norm = qini_diff / denom if abs(denom) > 1e-12 else np.nan
    qini_ratio = auc_model / auc_random if abs(auc_random) > 1e-12 else np.nan
    return {
        "ate": ate,
        "auc_model": auc_model,
        "auc_random": auc_random,
        "auc_perfect": auc_perfect,
        "qini_diff": qini_diff,
        "qini_norm": qini_norm,
        "qini_ratio": qini_ratio,
    }


def stratify_key(w_event, y):
    return (np.asarray(w_event, dtype=int) * 2 + np.asarray(y, dtype=int)).astype(int)


def make_fold_id(w_event, y):
    key = stratify_key(w_event, y)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold_id = np.zeros(len(y), dtype=int)
    dummy_x = np.zeros(len(y))
    for fold, (_, va) in enumerate(skf.split(dummy_x, key), 1):
        fold_id[va] = fold
    return fold_id


def rank01(a):
    a = np.asarray(a, dtype=float)
    return rankdata(a, method="average") / len(a)


def rank_average(score_list, weights=None):
    score_list = [np.asarray(s, dtype=float) for s in score_list if s is not None]
    if not score_list:
        return None
    ranks = np.vstack([rank01(s) for s in score_list])
    if weights is None:
        return ranks.mean(axis=0)
    weights = np.asarray(weights, dtype=float)
    if len(weights) != len(score_list) or weights.sum() <= 0:
        return ranks.mean(axis=0)
    weights = weights / weights.sum()
    return np.average(ranks, axis=0, weights=weights)


fold_ids = {
    "cash": make_fold_id(w_cash, Y_cash),
    "card": make_fold_id(w_card, Y_card),
}

print("Fold distribution cash target:", pd.Series(fold_ids["cash"]).value_counts().sort_index().to_dict())
print("Fold distribution card target:", pd.Series(fold_ids["card"]).value_counts().sort_index().to_dict())


Fold distribution cash target: {1: 27399, 2: 27399, 3: 27398, 4: 27398, 5: 27398}
Fold distribution card target: {1: 27399, 2: 27399, 3: 27398, 4: 27398, 5: 27398}


In [7]:
# ============================================================
# 2.1 Fold-safe uplift-aware feature selection — Chi filter
# ============================================================
# Paper method: bin-based divergence filter, Chi score = weighted chi-squared
# divergence between treatment conversion rate and control conversion rate across feature bins.
# Important: selection is done inside each fold using TRAIN indices only.

UPLIFT_FS_ENABLED = True
UPLIFT_FS_METHOD = "CHI"
UPLIFT_FS_TOPK_PER_TARGET = 40      # chỉnh K tại đây nếu muốn thử 20/30/40/50
UPLIFT_FS_N_BINS = 20
UPLIFT_FS_MIN_BIN_SIZE = 20
UPLIFT_FS_CHI_EPS = 1e-6            # smoothing để tránh chia 0 khi q gần 0 hoặc 1
UPLIFT_FS_USE_UNION_CASH_CARD = True  # top-K cash + top-K card -> union per fold
UPLIFT_FS_VERBOSE = True

_UPLIFT_FS_TARGET_CACHE = {}
_UPLIFT_FS_FOLD_CACHE = {}
_UPLIFT_FS_PRINTED = set()
UPLIFT_FS_LOGS = []


def _as_float_array(a):
    return np.asarray(a, dtype=np.float64)


def _make_feature_bins(x, n_bins=10):
    """Return integer bin codes for one feature. NaN/inf are put in a separate bin."""
    x = _as_float_array(x)
    n = len(x)
    codes = np.full(n, -1, dtype=np.int32)
    finite = np.isfinite(x)

    if finite.sum() == 0:
        return None

    xf = x[finite]
    uniq = np.unique(xf)
    if len(uniq) <= 1:
        return None

    # For binary / low-cardinality one-hot or label-encoded features, use exact value groups.
    if len(uniq) <= n_bins:
        _, inv = np.unique(xf, return_inverse=True)
        codes[finite] = inv.astype(np.int32)
        next_code = int(inv.max()) + 1
    else:
        # Equal-frequency bins are preferred by the paper for continuous features.
        try:
            q = min(n_bins, len(uniq))
            b = pd.qcut(xf, q=q, labels=False, duplicates="drop")
            b = np.asarray(b, dtype=np.int32)
            codes[finite] = b
            next_code = int(np.max(b)) + 1
        except Exception:
            # Robust fallback: rank-based equal-frequency bins.
            order = np.argsort(xf, kind="mergesort")
            b = np.empty(len(xf), dtype=np.int32)
            b[order] = np.floor(np.arange(len(xf)) * n_bins / max(len(xf), 1)).astype(np.int32)
            b = np.clip(b, 0, n_bins - 1)
            codes[finite] = b
            next_code = int(np.max(b)) + 1

    # Missing values are informative in credit data; keep them as a separate bin.
    if finite.sum() < n:
        codes[~finite] = next_code

    return codes


def _binary_chi_divergence(p_t, p_c, eps=1e-6):
    """Chi-square divergence for binary outcome: (p_t-p_c)^2/p_c + (p_t-p_c)^2/(1-p_c)."""
    p_c = float(np.clip(p_c, eps, 1.0 - eps))
    diff2 = float((p_t - p_c) ** 2)
    return diff2 / p_c + diff2 / (1.0 - p_c)


def chi_filter_scores(X_mat, y, w_event, row_idx, n_bins=10, min_bin_size=20, eps=1e-6):
    """Compute Chi-filter uplift feature score on row_idx only, no validation leakage."""
    X_sub = np.asarray(X_mat[row_idx], dtype=np.float32)
    y_sub = np.asarray(y[row_idx], dtype=np.float64)
    w_sub = np.asarray(w_event[row_idx], dtype=np.int8)
    n, p = X_sub.shape
    scores = np.zeros(p, dtype=np.float64)

    if n == 0 or p == 0:
        return scores

    for j in range(p):
        codes = _make_feature_bins(X_sub[:, j], n_bins=n_bins)
        if codes is None:
            continue
        score = 0.0
        for b in np.unique(codes):
            m = codes == b
            nb = int(m.sum())
            if nb < min_bin_size:
                continue
            mt = m & (w_sub == 1)
            mc = m & (w_sub == 0)
            nt = int(mt.sum())
            nc = int(mc.sum())
            if nt == 0 or nc == 0:
                continue
            p_t = float(y_sub[mt].mean())
            p_c = float(y_sub[mc].mean())
            score += (nb / n) * _binary_chi_divergence(p_t, p_c, eps=eps)
        scores[j] = score
    return scores


def _get_y_w_for_fs(target_name):
    if target_name == "cash":
        return Y_cash, w_cash
    if target_name == "card":
        return Y_card, w_card
    raise ValueError(target_name)


def _row_idx_signature(row_idx):
    """Small deterministic signature so target-cache never crosses different fold splits."""
    arr = np.asarray(row_idx, dtype=np.int64)
    if len(arr) == 0:
        return (0, 0, 0, 0)
    return (int(len(arr)), int(arr[0]), int(arr[-1]), int(arr.sum()))


def _get_target_chi_selection(target_name, fold, tr):
    """Top-K Chi features for one target/fold, fitted on train fold only."""
    key = (target_name, int(fold), _row_idx_signature(tr))
    if key in _UPLIFT_FS_TARGET_CACHE:
        return _UPLIFT_FS_TARGET_CACHE[key]

    y, w_event = _get_y_w_for_fs(target_name)
    scores = chi_filter_scores(
        X,
        y,
        w_event,
        tr,
        n_bins=UPLIFT_FS_N_BINS,
        min_bin_size=UPLIFT_FS_MIN_BIN_SIZE,
        eps=UPLIFT_FS_CHI_EPS,
    )
    k = min(int(UPLIFT_FS_TOPK_PER_TARGET), len(FEAT_COLS))
    order = np.argsort(-scores, kind="mergesort")
    selected_idx = order[:k]
    result = {"scores": scores, "selected_idx": selected_idx, "order": order}
    _UPLIFT_FS_TARGET_CACHE[key] = result
    return result


def get_fold_feature_indices(target_name, fold, tr):
    """Return feature indices selected for this fold. Uses train fold labels only."""
    if not UPLIFT_FS_ENABLED:
        return np.arange(len(FEAT_COLS), dtype=np.int64)

    # Cache key phải gồm chữ ký của tr: selection phụ thuộc vào index train,
    # không chỉ (target, fold). Tránh trả về feature cũ nếu cùng (target, fold)
    # nhưng tr khác (nested/repeated CV, refit full data...).
    key = (target_name, int(fold), bool(UPLIFT_FS_USE_UNION_CASH_CARD), _row_idx_signature(tr))
    if key in _UPLIFT_FS_FOLD_CACHE:
        return _UPLIFT_FS_FOLD_CACHE[key]

    if UPLIFT_FS_USE_UNION_CASH_CARD:
        # CHỦ ĐÍCH FOLD-SAFE (không phải bug):
        # Chi của CẢ HAI target đều tính trên `tr` của runner hiện tại
        # (train-fold của target đang chạy), KHÔNG dùng fold split riêng
        # của target kia. Lý do: cash và card có fold split khác nhau, nếu
        # dùng fold split của target kia thì nhãn của các dòng nằm trong
        # validation-fold của target hiện tại sẽ lọt vào bước chọn feature
        # -> rò rỉ. Đánh đổi: tập union cash/card có thể khác nhau giữa lúc
        # chạy runner cash và runner card. Đây là giá phải trả để giữ fold-safe;
        # union nhất quán tuyệt đối là bất khả thi khi hai target dùng split riêng.
        cash_res = _get_target_chi_selection("cash", fold, tr)
        card_res = _get_target_chi_selection("card", fold, tr)
        union_idx = np.unique(np.r_[cash_res["selected_idx"], card_res["selected_idx"]]).astype(np.int64)
        # Order by the stronger Chi score across two targets so the selected matrix is deterministic.
        combined_score = np.maximum(cash_res["scores"], card_res["scores"])
        selected_idx = np.array(sorted(union_idx, key=lambda j: combined_score[j], reverse=True), dtype=np.int64)
    else:
        selected_idx = _get_target_chi_selection(target_name, fold, tr)["selected_idx"].astype(np.int64)

    _UPLIFT_FS_FOLD_CACHE[key] = selected_idx

    selected_names = [FEAT_COLS[i] for i in selected_idx]
    UPLIFT_FS_LOGS.append({
        "target_runner": target_name,
        "fold": int(fold),
        "method": UPLIFT_FS_METHOD,
        "topk_per_target": int(UPLIFT_FS_TOPK_PER_TARGET),
        "use_union_cash_card": bool(UPLIFT_FS_USE_UNION_CASH_CARD),
        "n_selected_features": int(len(selected_idx)),
        "selected_features": ",".join(selected_names),
    })
    return selected_idx


def get_fold_feature_matrices(target_name, fold, tr, va, X_test_local=None):
    """Slice X/X_test by fold-safe Chi-selected features."""
    selected_idx = get_fold_feature_indices(target_name, fold, tr)
    Xtr = X[tr][:, selected_idx]
    Xva = X[va][:, selected_idx]
    Xte = X_test_local[:, selected_idx] if X_test_local is not None else None

    print_key = (target_name, int(fold))
    if UPLIFT_FS_VERBOSE and print_key not in _UPLIFT_FS_PRINTED:
        _UPLIFT_FS_PRINTED.add(print_key)
        names_preview = [FEAT_COLS[i] for i in selected_idx[:12]]
        print(
            f"  [CHI-FS] {target_name.upper()} fold {fold}: "
            f"selected {len(selected_idx)}/{len(FEAT_COLS)} features | top preview: {names_preview}"
        )
    return Xtr, Xva, Xte, selected_idx


print(
    "Uplift feature selection:",
    UPLIFT_FS_METHOD,
    "| enabled=", UPLIFT_FS_ENABLED,
    "| fold-safe=True",
    "| topK/target=", UPLIFT_FS_TOPK_PER_TARGET,
    "| union_cash_card=", UPLIFT_FS_USE_UNION_CASH_CARD,
    "| Chi_eps=", UPLIFT_FS_CHI_EPS,
)


Uplift feature selection: CHI | enabled= True | fold-safe=True | topK/target= 40 | union_cash_card= True | Chi_eps= 1e-06


## 3. Model factory

Base learner gồm KAN ban đầu (`kan`), Deep KAN pykan (`deep_kan`), MLP, SVM, AdaBoost, Decision Tree, RandomForest, ExtraTrees, LGBM, Logistic/Ridge, KNN, Fuzzy KNN, HistGradientBoosting, CatBoost, XGBoost.

In [8]:
class ConstantClassifier:
    def __init__(self, value):
        self.value = float(value)
        self.classes_ = np.array([0, 1])
    def fit(self, X, y, sample_weight=None):
        return self
    def predict_proba(self, X):
        p = np.full(len(X), self.value, dtype=float)
        p = np.clip(p, 0.0, 1.0)
        return np.c_[1 - p, p]


class ConstantRegressor:
    def __init__(self, value):
        self.value = float(value)
    def fit(self, X, y, sample_weight=None):
        return self
    def predict(self, X):
        return np.full(len(X), self.value, dtype=float)


def sigmoid_safe(z):
    z = np.clip(np.asarray(z, dtype=float), -35, 35)
    return 1.0 / (1.0 + np.exp(-z))


def stratified_subsample_indices(y, max_rows, seed=SEED):
    y = np.asarray(y)
    n = len(y)
    if max_rows is None or n <= max_rows:
        return np.arange(n)
    rng = np.random.default_rng(seed)
    uniq = np.unique(y)
    if len(uniq) <= 10 and set(np.unique(y.astype(int))).issubset({0, 1}):
        pos_idx = np.where(y.astype(int) == 1)[0]
        neg_idx = np.where(y.astype(int) == 0)[0]
        keep_pos = min(len(pos_idx), max_rows // 2)
        keep_neg = max_rows - keep_pos
        pos_sel = rng.choice(pos_idx, size=keep_pos, replace=False) if len(pos_idx) > keep_pos else pos_idx
        neg_sel = rng.choice(neg_idx, size=min(keep_neg, len(neg_idx)), replace=False)
        sel = np.r_[pos_sel, neg_sel]
        rng.shuffle(sel)
        return sel
    sel = rng.choice(np.arange(n), size=max_rows, replace=False)
    return np.sort(sel)


class DistanceKNNClassifier(BaseEstimator, ClassifierMixin):
    """KNN classifier có subsampling + inverse-distance probability để chạy được trên dữ liệu lớn."""
    def __init__(self, n_neighbors=75, max_train_rows=35000, power=1.0, seed=SEED, n_jobs=1):
        self.n_neighbors = n_neighbors
        self.max_train_rows = max_train_rows
        self.power = power
        self.seed = seed
        self.n_jobs = n_jobs
    def fit(self, X, y, sample_weight=None):
        idx = stratified_subsample_indices(y, self.max_train_rows, self.seed)
        self.X_ = np.asarray(X, dtype=np.float32)[idx]
        self.y_ = np.asarray(y, dtype=float)[idx]
        if sample_weight is None:
            self.sw_ = np.ones(len(idx), dtype=float)
        else:
            self.sw_ = np.asarray(sample_weight, dtype=float)[idx]
        self.classes_ = np.array([0, 1])
        k = min(int(self.n_neighbors), len(self.X_))
        self.nn_ = NearestNeighbors(n_neighbors=k, algorithm="auto", metric="minkowski", n_jobs=self.n_jobs)
        self.nn_.fit(self.X_)
        return self
    def predict_proba(self, X):
        dist, ind = self.nn_.kneighbors(np.asarray(X, dtype=np.float32), return_distance=True)
        weights = 1.0 / np.power(dist + 1e-6, self.power)
        weights = weights * self.sw_[ind]
        yy = self.y_[ind]
        p = (weights * yy).sum(axis=1) / np.maximum(weights.sum(axis=1), 1e-12)
        p = np.clip(p, 0.0, 1.0)
        return np.c_[1 - p, p]


class DistanceKNNRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, n_neighbors=75, max_train_rows=35000, power=1.0, seed=SEED, n_jobs=1):
        self.n_neighbors = n_neighbors
        self.max_train_rows = max_train_rows
        self.power = power
        self.seed = seed
        self.n_jobs = n_jobs
    def fit(self, X, y, sample_weight=None):
        idx = stratified_subsample_indices(y, self.max_train_rows, self.seed)
        self.X_ = np.asarray(X, dtype=np.float32)[idx]
        self.y_ = np.asarray(y, dtype=float)[idx]
        if sample_weight is None:
            self.sw_ = np.ones(len(idx), dtype=float)
        else:
            self.sw_ = np.asarray(sample_weight, dtype=float)[idx]
        k = min(int(self.n_neighbors), len(self.X_))
        self.nn_ = NearestNeighbors(n_neighbors=k, algorithm="auto", metric="minkowski", n_jobs=self.n_jobs)
        self.nn_.fit(self.X_)
        return self
    def predict(self, X):
        dist, ind = self.nn_.kneighbors(np.asarray(X, dtype=np.float32), return_distance=True)
        weights = 1.0 / np.power(dist + 1e-6, self.power)
        weights = weights * self.sw_[ind]
        yy = self.y_[ind]
        return (weights * yy).sum(axis=1) / np.maximum(weights.sum(axis=1), 1e-12)


class SplineKANClassifier(BaseEstimator, ClassifierMixin):
    """
    KAN ban đầu: mỗi feature được biến đổi qua spline basis univariate,
    sau đó học hàm cộng bằng LogisticRegression. Đây là KAN-style nhẹ, không cần pykan/torch.
    """
    def __init__(self, n_knots=6, degree=3, C=1.0, max_iter=700, seed=SEED):
        self.n_knots = n_knots
        self.degree = degree
        self.C = C
        self.max_iter = max_iter
        self.seed = seed
    def fit(self, X, y, sample_weight=None):
        self.spline_ = SplineTransformer(n_knots=self.n_knots, degree=self.degree, include_bias=False)
        Xb = self.spline_.fit_transform(np.asarray(X, dtype=np.float32))
        self.scaler_ = StandardScaler(with_mean=False)
        Xb = self.scaler_.fit_transform(Xb)
        self.model_ = LogisticRegression(
            C=self.C,
            penalty="l2",
            solver="saga",
            max_iter=self.max_iter,
            n_jobs=N_JOBS,
            random_state=self.seed,
        )
        self.model_.fit(Xb, y, sample_weight=sample_weight)
        self.classes_ = self.model_.classes_
        return self
    def predict_proba(self, X):
        Xb = self.spline_.transform(np.asarray(X, dtype=np.float32))
        Xb = self.scaler_.transform(Xb)
        proba = self.model_.predict_proba(Xb)
        if proba.shape[1] == 1:
            p = np.full(len(X), float(self.classes_[0] == 1))
            return np.c_[1 - p, p]
        return proba


class SplineKANRegressor(BaseEstimator, RegressorMixin):
    """KAN ban đầu dạng spline regressor dùng cho pseudo-outcome liên tục trong TO/X/R/DR learner."""
    def __init__(self, n_knots=6, degree=3, alpha=1.0):
        self.n_knots = n_knots
        self.degree = degree
        self.alpha = alpha
    def fit(self, X, y, sample_weight=None):
        self.spline_ = SplineTransformer(n_knots=self.n_knots, degree=self.degree, include_bias=False)
        Xb = self.spline_.fit_transform(np.asarray(X, dtype=np.float32))
        self.scaler_ = StandardScaler(with_mean=False)
        Xb = self.scaler_.fit_transform(Xb)
        self.model_ = Ridge(alpha=self.alpha, random_state=SEED)
        self.model_.fit(Xb, y, sample_weight=sample_weight)
        return self
    def predict(self, X):
        Xb = self.spline_.transform(np.asarray(X, dtype=np.float32))
        Xb = self.scaler_.transform(Xb)
        return self.model_.predict(Xb)


class DeepPyKANBase:
    """Base wrapper dùng pykan.KAN thật, train bằng PyTorch loop để hỗ trợ classification/regression trong sklearn-like API."""
    def __init__(
        self,
        hidden=None,
        grid=3,
        k=3,
        steps=80,
        batch_size=1024,
        lr=1e-3,
        weight_decay=1e-4,
        max_train_rows=12000,
        task="classification",
        seed=SEED,
        use_gpu=True,
        verbose=False,
    ):
        self.hidden = hidden if hidden is not None else [16, 8]
        self.grid = grid
        self.k = k
        self.steps = steps
        self.batch_size = batch_size
        self.lr = lr
        self.weight_decay = weight_decay
        self.max_train_rows = max_train_rows
        self.task = task
        self.seed = seed
        self.use_gpu = use_gpu
        self.verbose = verbose

    def _device(self):
        if self.use_gpu and torch.cuda.is_available():
            return torch.device("cuda")
        return torch.device("cpu")

    def _build_model(self, input_dim, device):
        width = [int(input_dim)] + [int(x) for x in self.hidden] + [1]
        # pykan API có thay đổi nhẹ giữa các version; thử một vài cách khởi tạo an toàn.
        constructors = [
            lambda: PyKAN(width=width, grid=self.grid, k=self.k, seed=self.seed, device=device),
            lambda: PyKAN(width=width, grid=self.grid, k=self.k, seed=self.seed),
            lambda: PyKAN(width=width, grid=self.grid, k=self.k),
        ]
        last_err = None
        for fn in constructors:
            try:
                model = fn()
                try:
                    model.to(device)
                except Exception:
                    pass
                # Tắt symbolic branch nếu version pykan có hỗ trợ, giúp train nhanh hơn trên tabular.
                try:
                    model.speed()
                except Exception:
                    pass
                return model
            except Exception as e:
                last_err = e
        raise last_err

    def _prepare_data(self, X, y, sample_weight=None):
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32).reshape(-1, 1)
        idx = stratified_subsample_indices(y.ravel(), self.max_train_rows, self.seed)
        Xs = X[idx]
        ys = y[idx]
        if sample_weight is None:
            sw = np.ones(len(idx), dtype=np.float32).reshape(-1, 1)
        else:
            sw = np.asarray(sample_weight, dtype=np.float32).reshape(-1, 1)[idx]
        self.scaler_ = StandardScaler()
        Xs = self.scaler_.fit_transform(Xs).astype(np.float32)
        return Xs, ys.astype(np.float32), sw.astype(np.float32)

    def _forward_tensor(self, xb):
        out = self.model_(xb)
        if isinstance(out, (tuple, list)):
            out = out[0]
        return out.reshape(-1, 1)

    def fit(self, X, y, sample_weight=None):
        rng = np.random.default_rng(self.seed)
        try:
            torch.manual_seed(self.seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(self.seed)
        except Exception:
            pass

        Xs, ys, sw = self._prepare_data(X, y, sample_weight=sample_weight)
        device = self._device()
        self.model_ = self._build_model(Xs.shape[1], device)
        self.input_dim_ = Xs.shape[1]

        xt = torch.tensor(Xs, dtype=torch.float32, device=device)
        yt = torch.tensor(ys, dtype=torch.float32, device=device)
        wt = torch.tensor(sw, dtype=torch.float32, device=device)

        optimizer = torch.optim.AdamW(self.model_.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        n = len(xt)
        bs = min(int(self.batch_size), n)
        pos = max(float(ys.sum()), 1.0)
        neg = max(float(len(ys) - ys.sum()), 1.0)
        pos_weight = torch.tensor([neg / pos], dtype=torch.float32, device=device)

        self.model_.train()
        for step in range(int(self.steps)):
            perm = rng.permutation(n)
            losses = []
            for start in range(0, n, bs):
                ids = torch.tensor(perm[start:start + bs], dtype=torch.long, device=device)
                xb = xt[ids]
                yb = yt[ids]
                wb = wt[ids]
                optimizer.zero_grad(set_to_none=True)
                out = self._forward_tensor(xb)
                if self.task == "classification":
                    loss_raw = torch.nn.functional.binary_cross_entropy_with_logits(out, yb, pos_weight=pos_weight, reduction="none")
                else:
                    loss_raw = torch.square(out - yb)
                loss = (loss_raw * wb).mean()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model_.parameters(), max_norm=5.0)
                optimizer.step()
                losses.append(float(loss.detach().cpu()))
            if self.verbose and (step == 0 or (step + 1) % 20 == 0 or step + 1 == int(self.steps)):
                print(f"    DeepKAN step {step+1}/{self.steps} loss={np.mean(losses):.6f}")
        return self

    def _predict_raw(self, X):
        X = np.asarray(X, dtype=np.float32)
        Xs = self.scaler_.transform(X).astype(np.float32)
        device = self._device()
        self.model_.eval()
        outs = []
        bs = max(1024, int(self.batch_size))
        with torch.no_grad():
            for start in range(0, len(Xs), bs):
                xb = torch.tensor(Xs[start:start + bs], dtype=torch.float32, device=device)
                out = self._forward_tensor(xb)
                outs.append(out.detach().cpu().numpy().reshape(-1))
        return np.concatenate(outs, axis=0)


class DeepPyKANClassifier(DeepPyKANBase, ClassifierMixin):
    def __init__(self, **kwargs):
        super().__init__(task="classification", **kwargs)
        self.classes_ = np.array([0, 1])
    def predict_proba(self, X):
        raw = self._predict_raw(X)
        p = sigmoid_safe(raw)
        p = np.clip(p, 0.0, 1.0)
        return np.c_[1 - p, p]


class DeepPyKANRegressor(DeepPyKANBase, RegressorMixin):
    def __init__(self, **kwargs):
        super().__init__(task="regression", **kwargs)
    def predict(self, X):
        return self._predict_raw(X).astype(float)


def scaled(estimator):
    return Pipeline([("scaler", StandardScaler()), ("model", estimator)])


def make_classifier(kind, seed=SEED):
    if kind == "lgbm":
        return LGBMClassifier(
            n_estimators=LGB_ESTIMATORS,
            learning_rate=0.035,
            num_leaves=31,
            max_depth=-1,
            min_child_samples=80,
            subsample=0.85,
            subsample_freq=1,
            colsample_bytree=0.85,
            reg_lambda=4.0,
            reg_alpha=0.5,
            objective="binary",
            random_state=seed,
            n_jobs=N_JOBS,
            verbose=-1,
            force_col_wise=True,
        )
    if kind == "cat":
        return CatBoostClassifier(
            iterations=CAT_ITERATIONS,
            learning_rate=0.045,
            depth=6,
            l2_leaf_reg=8.0,
            loss_function="Logloss",
            eval_metric="Logloss",
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
            thread_count=N_JOBS,
        )
    if kind == "xgb":
        return XGBClassifier(
            n_estimators=XGB_ESTIMATORS,
            learning_rate=0.04,
            max_depth=4,
            min_child_weight=80,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=5.0,
            reg_alpha=0.5,
            objective="binary:logistic",
            tree_method="hist",
            eval_metric="logloss",
            random_state=seed,
            n_jobs=N_JOBS,
            verbosity=0,
        )
    if kind == "kan":
        return SplineKANClassifier(n_knots=6, degree=3, C=0.8, max_iter=500, seed=seed)
    if kind == "deep_kan":
        return DeepPyKANClassifier(
            hidden=DEEP_KAN_WIDTH_HIDDEN,
            grid=DEEP_KAN_GRID,
            k=DEEP_KAN_K,
            steps=DEEP_KAN_STEPS,
            batch_size=DEEP_KAN_BATCH_SIZE,
            lr=DEEP_KAN_LR,
            weight_decay=DEEP_KAN_WEIGHT_DECAY,
            max_train_rows=DEEP_KAN_MAX_TRAIN_ROWS,
            seed=seed,
            use_gpu=DEEP_KAN_USE_GPU,
            verbose=False,
        )
    if kind == "mlp":
        return scaled(MLPClassifier(
            hidden_layer_sizes=(96, 48),
            activation="relu",
            alpha=1e-4,
            learning_rate_init=1e-3,
            max_iter=NN_MAX_ITER,
            tol=0.0,  # Không dùng early stopping; train tới max_iter.
            random_state=seed,
        ))
    if kind == "svm":
        return scaled(LinearSVC(C=0.5, class_weight="balanced", max_iter=4000, random_state=seed))
    if kind == "adaboost":
        return AdaBoostClassifier(n_estimators=max(80, TREE_ESTIMATORS // 2), learning_rate=0.04, random_state=seed)
    if kind == "dt":
        return DecisionTreeClassifier(
            max_depth=7,
            min_samples_leaf=120,
            min_samples_split=240,
            class_weight="balanced",
            random_state=seed,
        )
    if kind == "rf":
        return RandomForestClassifier(
            n_estimators=TREE_ESTIMATORS,
            max_depth=9,
            min_samples_leaf=80,
            max_features="sqrt",
            class_weight="balanced_subsample",
            n_jobs=N_JOBS,
            random_state=seed,
        )
    if kind == "extratrees":
        return ExtraTreesClassifier(
            n_estimators=TREE_ESTIMATORS,
            max_depth=10,
            min_samples_leaf=70,
            max_features="sqrt",
            class_weight="balanced",
            n_jobs=N_JOBS,
            random_state=seed,
        )
    if kind == "logistic":
        return scaled(LogisticRegression(
            C=0.7,
            penalty="l2",
            solver="saga",
            class_weight="balanced",
            max_iter=900,
            n_jobs=N_JOBS,
            random_state=seed,
        ))
    if kind == "knn":
        return scaled(DistanceKNNClassifier(
            n_neighbors=KNN_NEIGHBORS,
            max_train_rows=KNN_MAX_TRAIN_ROWS,
            power=1.0,
            seed=seed,
            n_jobs=N_JOBS,
        ))
    if kind == "fuzzy_knn":
        return scaled(DistanceKNNClassifier(
            n_neighbors=KNN_NEIGHBORS,
            max_train_rows=KNN_MAX_TRAIN_ROWS,
            power=FUZZY_KNN_POWER,
            seed=seed,
            n_jobs=N_JOBS,
        ))
    if kind == "hgb":
        return HistGradientBoostingClassifier(
            max_iter=max(120, TREE_ESTIMATORS // 2),
            learning_rate=0.045,
            max_leaf_nodes=31,
            min_samples_leaf=80,
            l2_regularization=1.0,
            random_state=seed,
        )
    raise ValueError(f"Unknown classifier kind: {kind}")


def make_regressor(kind, seed=SEED):
    if kind == "lgbm":
        return LGBMRegressor(
            n_estimators=LGB_ESTIMATORS,
            learning_rate=0.035,
            num_leaves=31,
            max_depth=-1,
            min_child_samples=80,
            subsample=0.85,
            subsample_freq=1,
            colsample_bytree=0.85,
            reg_lambda=4.0,
            reg_alpha=0.5,
            objective="regression",
            random_state=seed,
            n_jobs=N_JOBS,
            verbose=-1,
            force_col_wise=True,
        )
    if kind == "cat":
        return CatBoostRegressor(
            iterations=CAT_ITERATIONS,
            learning_rate=0.045,
            depth=6,
            l2_leaf_reg=8.0,
            loss_function="RMSE",
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
            thread_count=N_JOBS,
        )
    if kind == "xgb":
        return XGBRegressor(
            n_estimators=XGB_ESTIMATORS,
            learning_rate=0.04,
            max_depth=4,
            min_child_weight=80,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=5.0,
            reg_alpha=0.5,
            objective="reg:squarederror",
            tree_method="hist",
            random_state=seed,
            n_jobs=N_JOBS,
            verbosity=0,
        )
    if kind == "kan":
        return SplineKANRegressor(n_knots=6, degree=3, alpha=1.0)
    if kind == "deep_kan":
        return DeepPyKANRegressor(
            hidden=DEEP_KAN_WIDTH_HIDDEN,
            grid=DEEP_KAN_GRID,
            k=DEEP_KAN_K,
            steps=DEEP_KAN_STEPS,
            batch_size=DEEP_KAN_BATCH_SIZE,
            lr=DEEP_KAN_LR,
            weight_decay=DEEP_KAN_WEIGHT_DECAY,
            max_train_rows=DEEP_KAN_MAX_TRAIN_ROWS,
            seed=seed,
            use_gpu=DEEP_KAN_USE_GPU,
            verbose=False,
        )
    if kind == "mlp":
        return scaled(MLPRegressor(
            hidden_layer_sizes=(96, 48),
            activation="relu",
            alpha=1e-4,
            learning_rate_init=1e-3,
            max_iter=NN_MAX_ITER,
            tol=0.0,  # Không dùng early stopping; train tới max_iter.
            random_state=seed,
        ))
    if kind == "svm":
        return scaled(LinearSVR(C=0.5, epsilon=0.02, max_iter=4000, random_state=seed))
    if kind == "adaboost":
        return AdaBoostRegressor(n_estimators=max(80, TREE_ESTIMATORS // 2), learning_rate=0.04, random_state=seed)
    if kind == "dt":
        return DecisionTreeRegressor(
            max_depth=7,
            min_samples_leaf=120,
            min_samples_split=240,
            random_state=seed,
        )
    if kind == "rf":
        return RandomForestRegressor(
            n_estimators=TREE_ESTIMATORS,
            max_depth=9,
            min_samples_leaf=80,
            max_features="sqrt",
            n_jobs=N_JOBS,
            random_state=seed,
        )
    if kind == "extratrees":
        return ExtraTreesRegressor(
            n_estimators=TREE_ESTIMATORS,
            max_depth=10,
            min_samples_leaf=70,
            max_features="sqrt",
            n_jobs=N_JOBS,
            random_state=seed,
        )
    if kind == "logistic":
        # LogisticRegression không fit được pseudo-outcome liên tục; dùng Ridge làm linear counterpart cho TO/X/R/DR.
        return scaled(Ridge(alpha=2.0, random_state=seed))
    if kind == "knn":
        return scaled(DistanceKNNRegressor(
            n_neighbors=KNN_NEIGHBORS,
            max_train_rows=KNN_MAX_TRAIN_ROWS,
            power=1.0,
            seed=seed,
            n_jobs=N_JOBS,
        ))
    if kind == "fuzzy_knn":
        return scaled(DistanceKNNRegressor(
            n_neighbors=KNN_NEIGHBORS,
            max_train_rows=KNN_MAX_TRAIN_ROWS,
            power=FUZZY_KNN_POWER,
            seed=seed,
            n_jobs=N_JOBS,
        ))
    if kind == "hgb":
        return HistGradientBoostingRegressor(
            max_iter=max(120, TREE_ESTIMATORS // 2),
            learning_rate=0.045,
            max_leaf_nodes=31,
            min_samples_leaf=80,
            l2_regularization=1.0,
            random_state=seed,
        )
    raise ValueError(f"Unknown regressor kind: {kind}")


def class_weights(y, pos_weight=None):
    y = np.asarray(y, dtype=int)
    if pos_weight is None:
        pos = max(int(y.sum()), 1)
        neg = max(len(y) - pos, 1)
        pos_weight = neg / pos
    return np.where(y == 1, pos_weight, 1.0).astype(np.float32)


def common_pos_weight(y_treat, y_ctrl):
    def spw(y):
        y = np.asarray(y, dtype=int)
        pos = max(int(y.sum()), 1)
        neg = max(len(y) - pos, 1)
        return neg / pos
    return 0.5 * (spw(y_treat) + spw(y_ctrl))


def fit_with_optional_sample_weight(model, X_train, y_train, sample_weight=None):
    if sample_weight is None:
        model.fit(X_train, y_train)
        return model
    sample_weight = np.asarray(sample_weight, dtype=np.float32)
    try:
        if isinstance(model, Pipeline):
            model.fit(X_train, y_train, model__sample_weight=sample_weight)
        else:
            model.fit(X_train, y_train, sample_weight=sample_weight)
        return model
    except TypeError:
        model.fit(X_train, y_train)
        return model
    except ValueError as e:
        if "sample_weight" in str(e) or "Pipeline.fit" in str(e):
            model.fit(X_train, y_train)
            return model
        raise


def safe_fit_classifier(kind, X_train, y_train, sample_weight=None, seed=SEED):
    y_train = np.asarray(y_train, dtype=int)
    if len(y_train) == 0 or len(np.unique(y_train)) < 2:
        return ConstantClassifier(float(y_train.mean()) if len(y_train) else 0.0)
    model = make_classifier(kind, seed=seed)
    return fit_with_optional_sample_weight(model, X_train, y_train, sample_weight=sample_weight)


def safe_fit_regressor(kind, X_train, y_train, sample_weight=None, seed=SEED):
    y_train = np.asarray(y_train, dtype=float)
    if len(y_train) == 0 or np.nanstd(y_train) < 1e-12:
        return ConstantRegressor(float(np.nanmean(y_train)) if len(y_train) else 0.0)
    model = make_regressor(kind, seed=seed)
    return fit_with_optional_sample_weight(model, X_train, y_train, sample_weight=sample_weight)


def get_pipeline_final_estimator(model):
    if isinstance(model, Pipeline):
        return model.named_steps.get("model", model.steps[-1][1])
    return model


def predict_classifier(model, X_pred):
    # Ưu tiên probability; nếu SVM không có predict_proba thì dùng decision_function -> sigmoid để có score ranking.
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_pred)
        proba = np.asarray(proba)
        if proba.ndim == 1:
            return np.clip(proba.astype(float), 0.0, 1.0)
        if proba.shape[1] == 1:
            classes = getattr(model, "classes_", getattr(get_pipeline_final_estimator(model), "classes_", np.array([1])))
            return np.full(len(X_pred), float(classes[0] == 1))
        classes = getattr(model, "classes_", getattr(get_pipeline_final_estimator(model), "classes_", None))
        if classes is not None and 1 in list(classes):
            idx = list(classes).index(1)
        else:
            idx = -1
        return proba[:, idx].astype(float)
    if hasattr(model, "decision_function"):
        return sigmoid_safe(model.decision_function(X_pred))
    pred = model.predict(X_pred)
    return np.asarray(pred, dtype=float)


def predict_regressor(model, X_pred):
    pred = np.asarray(model.predict(X_pred), dtype=float)
    if pred.ndim > 1:
        pred = pred[:, 0]
    return pred


## 4. Uplift learner functions

Các base model được đưa vào 8 hướng meta-learner: T, S, Interaction S, Class Transformation, Transformed Outcome, X, R, DR. Direct uplift models như Uplift Tree/RF, Interaction Tree, Causal Inference Tree và UTBoost sẽ chạy nếu bật option và cài package tương ứng.

In [9]:

def fold_iter(target_name):
    fid = fold_ids[target_name]
    for fold in range(1, N_SPLITS + 1):
        va = np.where(fid == fold)[0]
        tr = np.where(fid != fold)[0]
        yield fold, tr, va


def get_target_arrays(target_name):
    if target_name == "cash":
        return Y_cash, w_cash, 1, "CASH"
    if target_name == "card":
        return Y_card, w_card, 0, "CARD"
    raise ValueError(target_name)


def print_fold_metric(prefix, target_name, y, w_event, va, pred):
    met = qini_metrics(y[va], w_event[va], pred)
    print(f"  fold {prefix}: Qini_coefficient_{target_name}={met['qini_norm']:+.5f} | Q_ratio={met['qini_ratio']:+.4f}")
    return met


def run_t_learner(kind, target_name, X_test=None):
    y, w_event, treat_value, label = get_target_arrays(target_name)
    oof = np.zeros(len(X), dtype=float)
    test_score = np.zeros(len(X_test), dtype=float) if X_test is not None else None
    print(f"\n[{label}] T-Learner {kind}")
    if target_name == "cash":
        print("  internal response models: Cash-treatment + Cash-label, Card-treatment + Cash-label")
    else:
        print("  internal response models: Card-treatment + Card-label, Cash-treatment + Card-label")

    for fold, tr, va in fold_iter(target_name):
        Xtr, Xva, Xte, _ = get_fold_feature_matrices(target_name, fold, tr, va, X_test)
        mask_t = w_event[tr] == 1
        mask_c = ~mask_t
        y_t = y[tr][mask_t]
        y_c = y[tr][mask_c]
        spw_common = common_pos_weight(y_t, y_c)
        m_t = safe_fit_classifier(kind, Xtr[mask_t], y_t, class_weights(y_t, spw_common), seed=SEED + fold)
        m_c = safe_fit_classifier(kind, Xtr[mask_c], y_c, class_weights(y_c, spw_common), seed=SEED + 100 + fold)
        pred = predict_classifier(m_t, Xva) - predict_classifier(m_c, Xva)
        oof[va] = pred
        if Xte is not None:
            test_score += (predict_classifier(m_t, Xte) - predict_classifier(m_c, Xte)) / N_SPLITS
        print_fold_metric(fold, target_name, y, w_event, va, pred)
        del m_t, m_c, Xtr, Xva, Xte
        gc.collect()
    full_met = qini_metrics(y, w_event, oof)
    print(f"  OOF: Qini_coefficient_{target_name}={full_met['qini_norm']:+.5f} | Q_ratio={full_met['qini_ratio']:+.4f}")
    return oof, test_score


def run_s_learner(kind, target_name, X_test=None):
    y, w_event, treat_value, label = get_target_arrays(target_name)
    oof = np.zeros(len(X), dtype=float)
    test_score = np.zeros(len(X_test), dtype=float) if X_test is not None else None
    print(f"\n[{label}] S-Learner {kind}")
    for fold, tr, va in fold_iter(target_name):
        Xtr, Xva, Xte, _ = get_fold_feature_matrices(target_name, fold, tr, va, X_test)
        Xtr_aug = np.c_[Xtr, w_event[tr].astype(np.float32)]
        Xva_event = np.c_[Xva, np.ones(len(va), dtype=np.float32)]
        Xva_control = np.c_[Xva, np.zeros(len(va), dtype=np.float32)]
        sw = class_weights(y[tr])
        m = safe_fit_classifier(kind, Xtr_aug, y[tr], sw, seed=SEED + fold)
        pred = predict_classifier(m, Xva_event) - predict_classifier(m, Xva_control)
        oof[va] = pred
        if Xte is not None:
            Xte_event = np.c_[Xte, np.ones(len(Xte), dtype=np.float32)]
            Xte_control = np.c_[Xte, np.zeros(len(Xte), dtype=np.float32)]
            test_score += (predict_classifier(m, Xte_event) - predict_classifier(m, Xte_control)) / N_SPLITS
        print_fold_metric(fold, target_name, y, w_event, va, pred)
        del m, Xtr, Xva, Xte
        gc.collect()
    full_met = qini_metrics(y, w_event, oof)
    print(f"  OOF: Qini_coefficient_{target_name}={full_met['qini_norm']:+.5f} | Q_ratio={full_met['qini_ratio']:+.4f}")
    return oof, test_score


def run_interaction_learner(kind, target_name, X_test=None):
    # Strong S-learner variant: one model with treatment flag and X*treatment interactions.
    y, w_event, treat_value, label = get_target_arrays(target_name)
    oof = np.zeros(len(X), dtype=float)
    test_score = np.zeros(len(X_test), dtype=float) if X_test is not None else None
    print(f"\n[{label}] Interaction S-Learner {kind}")
    for fold, tr, va in fold_iter(target_name):
        Xtr, Xva, Xte, _ = get_fold_feature_matrices(target_name, fold, tr, va, X_test)
        wt = w_event[tr].astype(np.float32)[:, None]
        Xtr_aug = np.c_[Xtr, wt, Xtr * wt]
        Xva_event = np.c_[Xva, np.ones((len(va), 1), dtype=np.float32), Xva]
        Xva_control = np.c_[Xva, np.zeros((len(va), 1), dtype=np.float32), np.zeros_like(Xva)]
        sw = class_weights(y[tr])
        m = safe_fit_classifier(kind, Xtr_aug, y[tr], sw, seed=SEED + fold)
        pred = predict_classifier(m, Xva_event) - predict_classifier(m, Xva_control)
        oof[va] = pred
        if Xte is not None:
            Xte_event = np.c_[Xte, np.ones((len(Xte), 1), dtype=np.float32), Xte]
            Xte_control = np.c_[Xte, np.zeros((len(Xte), 1), dtype=np.float32), np.zeros_like(Xte)]
            test_score += (predict_classifier(m, Xte_event) - predict_classifier(m, Xte_control)) / N_SPLITS
        print_fold_metric(fold, target_name, y, w_event, va, pred)
        del m, Xtr, Xva, Xte
        gc.collect()
    full_met = qini_metrics(y, w_event, oof)
    print(f"  OOF: Qini_coefficient_{target_name}={full_met['qini_norm']:+.5f} | Q_ratio={full_met['qini_ratio']:+.4f}")
    return oof, test_score


def run_class_transformation(kind, target_name, X_test=None):
    # Revert-label / Class Transformation: z=1 nếu treated&converted hoặc control&not converted.
    y, w_event, treat_value, label = get_target_arrays(target_name)
    z = np.where(w_event == 1, y, 1 - y).astype(int)
    oof = np.zeros(len(X), dtype=float)
    test_score = np.zeros(len(X_test), dtype=float) if X_test is not None else None
    print(f"\n[{label}] Class Transformation {kind}")
    for fold, tr, va in fold_iter(target_name):
        Xtr, Xva, Xte, _ = get_fold_feature_matrices(target_name, fold, tr, va, X_test)
        e = np.clip(w_event[tr].mean(), 1e-6, 1 - 1e-6)
        balance_w = np.where(w_event[tr] == 1, 1.0 / e, 1.0 / (1.0 - e)).astype(np.float32)
        sw = balance_w * class_weights(z[tr])
        m = safe_fit_classifier(kind, Xtr, z[tr], sw, seed=SEED + fold)
        pred = 2.0 * predict_classifier(m, Xva) - 1.0
        oof[va] = pred
        if Xte is not None:
            test_score += (2.0 * predict_classifier(m, Xte) - 1.0) / N_SPLITS
        print_fold_metric(fold, target_name, y, w_event, va, pred)
        del m, Xtr, Xva, Xte
        gc.collect()
    full_met = qini_metrics(y, w_event, oof)
    print(f"  OOF: Qini_coefficient_{target_name}={full_met['qini_norm']:+.5f} | Q_ratio={full_met['qini_ratio']:+.4f}")
    return oof, test_score


def run_transformed_outcome(kind, target_name, X_test=None):
    # y* = W*Y/e - (1-W)*Y/(1-e), E[y*|X] = treatment effect.
    y, w_event, treat_value, label = get_target_arrays(target_name)
    oof = np.zeros(len(X), dtype=float)
    test_score = np.zeros(len(X_test), dtype=float) if X_test is not None else None
    print(f"\n[{label}] Transformed Outcome {kind}")
    for fold, tr, va in fold_iter(target_name):
        Xtr, Xva, Xte, _ = get_fold_feature_matrices(target_name, fold, tr, va, X_test)
        e = np.clip(w_event[tr].mean(), 1e-6, 1 - 1e-6)
        y_star = w_event[tr] * y[tr] / e - (1 - w_event[tr]) * y[tr] / (1 - e)
        m = safe_fit_regressor(kind, Xtr, y_star, seed=SEED + fold)
        pred = predict_regressor(m, Xva)
        oof[va] = pred
        if Xte is not None:
            test_score += predict_regressor(m, Xte) / N_SPLITS
        print_fold_metric(fold, target_name, y, w_event, va, pred)
        del m, Xtr, Xva, Xte
        gc.collect()
    full_met = qini_metrics(y, w_event, oof)
    print(f"  OOF: Qini_coefficient_{target_name}={full_met['qini_norm']:+.5f} | Q_ratio={full_met['qini_ratio']:+.4f}")
    return oof, test_score


In [10]:

def run_x_learner(kind, target_name, X_test=None):
    # X-Learner: T-learner outcome models -> impute individual effects -> effect regressors.
    y, w_event, treat_value, label = get_target_arrays(target_name)
    oof = np.zeros(len(X), dtype=float)
    test_score = np.zeros(len(X_test), dtype=float) if X_test is not None else None
    print(f"\n[{label}] X-Learner {kind}")
    for fold, tr, va in fold_iter(target_name):
        Xtr, Xva, Xte, _ = get_fold_feature_matrices(target_name, fold, tr, va, X_test)
        mask_t = w_event[tr] == 1
        mask_c = ~mask_t
        y_t = y[tr][mask_t]
        y_c = y[tr][mask_c]
        spw_common = common_pos_weight(y_t, y_c)
        m_t = safe_fit_classifier(kind, Xtr[mask_t], y_t, class_weights(y_t, spw_common), seed=SEED + fold)
        m_c = safe_fit_classifier(kind, Xtr[mask_c], y_c, class_weights(y_c, spw_common), seed=SEED + 100 + fold)

        d_t = y_t - predict_classifier(m_c, Xtr[mask_t])
        d_c = predict_classifier(m_t, Xtr[mask_c]) - y_c

        tau_t = safe_fit_regressor(kind, Xtr[mask_t], d_t, seed=SEED + 200 + fold)
        tau_c = safe_fit_regressor(kind, Xtr[mask_c], d_c, seed=SEED + 300 + fold)

        e = np.clip(w_event[tr].mean(), 1e-6, 1 - 1e-6)
        pred = e * predict_regressor(tau_c, Xva) + (1.0 - e) * predict_regressor(tau_t, Xva)
        oof[va] = pred
        if Xte is not None:
            test_score += (e * predict_regressor(tau_c, Xte) + (1.0 - e) * predict_regressor(tau_t, Xte)) / N_SPLITS
        print_fold_metric(fold, target_name, y, w_event, va, pred)
        del m_t, m_c, tau_t, tau_c, Xtr, Xva, Xte
        gc.collect()
    full_met = qini_metrics(y, w_event, oof)
    print(f"  OOF: Qini_coefficient_{target_name}={full_met['qini_norm']:+.5f} | Q_ratio={full_met['qini_ratio']:+.4f}")
    return oof, test_score


def run_r_learner(kind, target_name, X_test=None):
    # R-Learner: residualize outcome and randomized treatment; no propensity model is trained.
    y, w_event, treat_value, label = get_target_arrays(target_name)
    oof = np.zeros(len(X), dtype=float)
    test_score = np.zeros(len(X_test), dtype=float) if X_test is not None else None
    print(f"\n[{label}] R-Learner {kind}")
    for fold, tr, va in fold_iter(target_name):
        Xtr, Xva, Xte, _ = get_fold_feature_matrices(target_name, fold, tr, va, X_test)
        e = np.clip(w_event[tr].mean(), 1e-6, 1 - 1e-6)
        m_y = safe_fit_classifier(kind, Xtr, y[tr], class_weights(y[tr]), seed=SEED + fold)
        mu_tr = predict_classifier(m_y, Xtr)
        denom = w_event[tr] - e
        denom_safe = np.where(np.abs(denom) < 1e-6, np.where(denom >= 0, 1e-6, -1e-6), denom)
        pseudo_tau = (y[tr] - mu_tr) / denom_safe
        pseudo_w = np.square(denom).astype(np.float32)
        tau_model = safe_fit_regressor(kind, Xtr, pseudo_tau, pseudo_w, seed=SEED + 100 + fold)
        pred = predict_regressor(tau_model, Xva)
        oof[va] = pred
        if Xte is not None:
            test_score += predict_regressor(tau_model, Xte) / N_SPLITS
        print_fold_metric(fold, target_name, y, w_event, va, pred)
        del m_y, tau_model, Xtr, Xva, Xte
        gc.collect()
    full_met = qini_metrics(y, w_event, oof)
    print(f"  OOF: Qini_coefficient_{target_name}={full_met['qini_norm']:+.5f} | Q_ratio={full_met['qini_ratio']:+.4f}")
    return oof, test_score


def run_dr_learner(kind, target_name, X_test=None):
    # DR-Learner: doubly-robust pseudo-outcome with constant randomized propensity.
    y, w_event, treat_value, label = get_target_arrays(target_name)
    oof = np.zeros(len(X), dtype=float)
    test_score = np.zeros(len(X_test), dtype=float) if X_test is not None else None
    print(f"\n[{label}] DR-Learner {kind}")
    for fold, tr, va in fold_iter(target_name):
        Xtr, Xva, Xte, _ = get_fold_feature_matrices(target_name, fold, tr, va, X_test)
        mask_t = w_event[tr] == 1
        mask_c = ~mask_t
        y_t = y[tr][mask_t]
        y_c = y[tr][mask_c]
        spw_common = common_pos_weight(y_t, y_c)
        m_t = safe_fit_classifier(kind, Xtr[mask_t], y_t, class_weights(y_t, spw_common), seed=SEED + fold)
        m_c = safe_fit_classifier(kind, Xtr[mask_c], y_c, class_weights(y_c, spw_common), seed=SEED + 100 + fold)

        mu1_tr = predict_classifier(m_t, Xtr)
        mu0_tr = predict_classifier(m_c, Xtr)
        e = np.clip(w_event[tr].mean(), 1e-6, 1 - 1e-6)
        phi = (
            mu1_tr - mu0_tr
            + w_event[tr] * (y[tr] - mu1_tr) / e
            - (1 - w_event[tr]) * (y[tr] - mu0_tr) / (1 - e)
        )
        tau_model = safe_fit_regressor(kind, Xtr, phi, seed=SEED + 200 + fold)
        pred = predict_regressor(tau_model, Xva)
        oof[va] = pred
        if Xte is not None:
            test_score += predict_regressor(tau_model, Xte) / N_SPLITS
        print_fold_metric(fold, target_name, y, w_event, va, pred)
        del m_t, m_c, tau_model, Xtr, Xva, Xte
        gc.collect()
    full_met = qini_metrics(y, w_event, oof)
    print(f"  OOF: Qini_coefficient_{target_name}={full_met['qini_norm']:+.5f} | Q_ratio={full_met['qini_ratio']:+.4f}")
    return oof, test_score


def extract_uplift_prediction(pred_raw):
    """Chuẩn hóa output từ causalml/utboost về vector uplift 1D."""
    if isinstance(pred_raw, pd.DataFrame):
        num = pred_raw.select_dtypes(include=[np.number])
        for c in ["uplift", "treatment_effect", "maxTreatmentEffect", "recommended_treatment_effect"]:
            if c in num.columns:
                return num[c].to_numpy(dtype=float)
        return num.iloc[:, -1].to_numpy(dtype=float)
    arr = np.asarray(pred_raw)
    if arr.ndim == 1:
        return arr.astype(float)
    if arr.shape[1] >= 2:
        return (arr[:, 1] - arr[:, 0]).astype(float)
    return arr[:, 0].astype(float)


def run_optional_causalml_model(model_type, criterion, target_name, X_test=None):
    # Direct Uplift Tree / Uplift Random Forest / Interaction Tree / Causal Inference Tree.
    if not AVAILABLE.get("causalml", False):
        raise ImportError("causalml is not installed")
    y, w_event, treat_value, label = get_target_arrays(target_name)
    oof = np.zeros(len(X), dtype=float)
    test_score = np.zeros(len(X_test), dtype=float) if X_test is not None else None
    pretty = "UpliftTree" if model_type == "tree" else "UpliftRandomForest"
    print(f"\n[{label}] CausalML {pretty} criterion={criterion}")
    for fold, tr, va in fold_iter(target_name):
        Xtr, Xva, Xte, _ = get_fold_feature_matrices(target_name, fold, tr, va, X_test)
        treatment_labels = np.where(w_event == 1, "treatment", "control")
        common_kwargs = dict(
            max_depth=5 if model_type == "forest" else 4,
            min_samples_leaf=120,
            min_samples_treatment=40,
            evaluationFunction=criterion,
            control_name="control",
            random_state=SEED + fold,
        )
        if model_type == "forest":
            model = UpliftRandomForestClassifier(
                n_estimators=80,
                max_features=min(12, Xtr.shape[1]),
                n_jobs=N_JOBS,
                **common_kwargs,
            )
        else:
            model = UpliftTreeClassifier(
                max_features=min(12, Xtr.shape[1]),
                **common_kwargs,
            )
        model.fit(Xtr, treatment=treatment_labels[tr], y=y[tr])
        pred = extract_uplift_prediction(model.predict(Xva))
        oof[va] = pred
        if Xte is not None:
            test_score += extract_uplift_prediction(model.predict(Xte)) / N_SPLITS
        print_fold_metric(fold, target_name, y, w_event, va, pred)
        del model, Xtr, Xva, Xte
        gc.collect()
    full_met = qini_metrics(y, w_event, oof)
    print(f"  OOF: Qini_coefficient_{target_name}={full_met['qini_norm']:+.5f} | Q_ratio={full_met['qini_ratio']:+.4f}")
    return oof, test_score


def run_optional_utboost(target_name, X_test=None):
    if not AVAILABLE.get("utboost", False):
        raise ImportError("utboost is not installed")
    y, w_event, treat_value, label = get_target_arrays(target_name)
    oof = np.zeros(len(X), dtype=float)
    test_score = np.zeros(len(X_test), dtype=float) if X_test is not None else None
    print(f"\n[{label}] UTBoost")
    for fold, tr, va in fold_iter(target_name):
        Xtr, Xva, Xte, _ = get_fold_feature_matrices(target_name, fold, tr, va, X_test)
        model = UTBClassifier(
            ensemble_type="boosting",
            criterion="gbm",
            iterations=UTB_ITERATIONS,
            max_depth=4,
        )
        model.fit(X=Xtr, ti=w_event[tr].astype(int), y=y[tr].astype(int))
        pred = extract_uplift_prediction(model.predict(Xva))
        oof[va] = pred
        if Xte is not None:
            test_score += extract_uplift_prediction(model.predict(Xte)) / N_SPLITS
        print_fold_metric(fold, target_name, y, w_event, va, pred)
        del model, Xtr, Xva, Xte
        gc.collect()
    full_met = qini_metrics(y, w_event, oof)
    print(f"  OOF: Qini_coefficient_{target_name}={full_met['qini_norm']:+.5f} | Q_ratio={full_met['qini_ratio']:+.4f}")
    return oof, test_score


## 5. Run all learners + direct uplift models

In [ ]:
scores_oof = {"cash": {}, "card": {}}
scores_test = {"cash": {}, "card": {}}
run_log = []

RUNNER_MAP = {
    "T": run_t_learner,
    "S": run_s_learner,
    "INT": run_interaction_learner,
    "CT": run_class_transformation,
    "TO": run_transformed_outcome,
    "X": run_x_learner,
    "R": run_r_learner,
    "DR": run_dr_learner,
}


def should_run_model(model_name):
    selected = globals().get("SELECTED_MODEL_NAMES", None)
    return (not selected) or (model_name in selected)



def summarize_one_model(model_name):
    row = {"model": model_name}
    if model_name in scores_oof["cash"]:
        row["Qini_coefficient_cash"] = qini_metrics(Y_cash, w_cash, scores_oof["cash"][model_name])["qini_norm"]
    else:
        row["Qini_coefficient_cash"] = np.nan
    if model_name in scores_oof["card"]:
        row["Qini_coefficient_card"] = qini_metrics(Y_card, w_card, scores_oof["card"][model_name])["qini_norm"]
    else:
        row["Qini_coefficient_card"] = np.nan
    row["Qini_coefficient_total"] = row["Qini_coefficient_cash"] + row["Qini_coefficient_card"]
    return pd.DataFrame([row])


def add_pair_result(model_name, cash_tuple, card_tuple):
    cash_oof, cash_te = cash_tuple
    card_oof, card_te = card_tuple
    scores_oof["cash"][model_name] = cash_oof
    scores_oof["card"][model_name] = card_oof
    if cash_te is not None:
        scores_test["cash"][model_name] = cash_te
    if card_te is not None:
        scores_test["card"][model_name] = card_te
    print(f"\n>>> Summary after {model_name}: CASH / CARD / TOTAL")
    display(summarize_one_model(model_name).round(6))


def run_pair(learner_key, kind):
    model_name = f"{learner_key}_{kind.upper()}"
    runner = RUNNER_MAP[learner_key]
    t0 = time.time()
    cash_tuple = runner(kind, "cash", X_test=X_test)
    card_tuple = runner(kind, "card", X_test=X_test)
    add_pair_result(model_name, cash_tuple, card_tuple)
    run_log.append({"model": model_name, "minutes": (time.time() - t0) / 60, "type": "meta_learner"})


def causalml_tree_model_name(model_type, criterion):
    # Dùng naming thống nhất với bảng top model: UPLIFT_TREE_CIT / UPLIFT_TREE_IT / UPLIFT_RF_*
    prefix = "UPLIFT_TREE" if model_type == "tree" else "UPLIFT_RF"
    return f"{prefix}_{str(criterion).upper()}"


def run_direct_causalml_pair(model_type, criterion):
    model_name = causalml_tree_model_name(model_type, criterion)
    t0 = time.time()
    cash_tuple = run_optional_causalml_model(model_type, criterion, "cash", X_test=X_test)
    card_tuple = run_optional_causalml_model(model_type, criterion, "card", X_test=X_test)
    add_pair_result(model_name, cash_tuple, card_tuple)
    run_log.append({"model": model_name, "minutes": (time.time() - t0) / 60, "type": "direct_uplift_causalml"})


def run_utboost_pair():
    model_name = "UTBOOST"
    t0 = time.time()
    cash_tuple = run_optional_utboost("cash", X_test=X_test)
    card_tuple = run_optional_utboost("card", X_test=X_test)
    add_pair_result(model_name, cash_tuple, card_tuple)
    run_log.append({"model": model_name, "minutes": (time.time() - t0) / 60, "type": "direct_utboost"})


start_all = time.time()

print("\n========== RUN META-LEARNER MODELS ==========")
print("Expected meta model names:")
print([m for m in globals().get("TOP_MODEL_NAMES", []) if any(m.startswith(lk + "_") for lk in RUNNER_MAP)])

for learner_key in RUN_LEARNERS:
    if learner_key not in RUNNER_MAP:
        continue
    for kind in RUN_MODEL_KINDS:
        try:
            model_name = f"{learner_key}_{kind.upper()}"
            if not should_run_model(model_name):
                continue
            run_pair(learner_key, kind)
        except Exception as e:
            model_name = f"{learner_key}_{kind.upper()}"
            print(f"\n!!! Skip {model_name} due to error: {repr(e)}")
            run_log.append({"model": model_name, "minutes": np.nan, "type": "meta_learner", "error": repr(e)})
            gc.collect()

print("\n========== RUN DIRECT UPLIFT TREE/RF MODELS ==========")
if RUN_OPTIONAL_CAUSALML_TREE:
    if not AVAILABLE.get("causalml", False):
        print("RUN_OPTIONAL_CAUSALML_TREE=True nhưng causalml chưa được cài -> skip direct uplift tree/RF/IT/CIT.")
    else:
        for criterion in CAUSALML_TREE_CRITERIA:
            try:
                model_name = causalml_tree_model_name("tree", criterion)
                if not should_run_model(model_name):
                    continue
                run_direct_causalml_pair("tree", criterion)
            except Exception as e:
                model_name = causalml_tree_model_name("tree", criterion)
                print(f"Skip {model_name} due to error: {repr(e)}")
                run_log.append({"model": model_name, "minutes": np.nan, "type": "direct_uplift_causalml", "error": repr(e)})
                gc.collect()
        for criterion in CAUSALML_FOREST_CRITERIA:
            try:
                model_name = causalml_tree_model_name("forest", criterion)
                if not should_run_model(model_name):
                    continue
                run_direct_causalml_pair("forest", criterion)
            except Exception as e:
                model_name = causalml_tree_model_name("forest", criterion)
                print(f"Skip {model_name} due to error: {repr(e)}")
                run_log.append({"model": model_name, "minutes": np.nan, "type": "direct_uplift_causalml", "error": repr(e)})
                gc.collect()
else:
    print("RUN_OPTIONAL_CAUSALML_TREE=False -> skip direct uplift tree/RF/IT/CIT.")

print("\n========== RUN UTBOOST ==========")
if RUN_OPTIONAL_UTBOOST:
    if not AVAILABLE.get("utboost", False):
        print("RUN_OPTIONAL_UTBOOST=True nhưng utboost chưa được cài -> skip UTBoost.")
    else:
        try:
            if should_run_model("UTBOOST"):
                run_utboost_pair()
        except Exception as e:
            print(f"Skip UTBOOST due to error: {repr(e)}")
            run_log.append({"model": "UTBOOST", "minutes": np.nan, "type": "direct_utboost", "error": repr(e)})
            gc.collect()
else:
    print("RUN_OPTIONAL_UTBOOST=False -> skip UTBoost.")

print(f"\nTotal model runtime: {(time.time() - start_all)/60:.1f} minutes")
print("Models finished:", list(scores_oof["cash"].keys()))
print("Total finished models:", len(scores_oof["cash"]))



========== RUN META-LEARNER MODELS ==========
Expected meta model names:
['INT_RF']

[CASH] Interaction S-Learner rf
  [CHI-FS] CASH fold 1: selected 49/62 features | top preview: ['cnt_fi_inst_appl_12m', 'cnt_lender_has2pay_last_24m', 'cnt_earlyterminated_cash', 'last_offer_limit', 'cnt_month_from_last_appl', 'cnt_total_payment', 'cli_emp_period', 'cnt_month_status1', 'cli_age', 'avg_sign_vol', 'cnt_fail_tls_3m', 'min_sign_vol']
  fold 1: Qini_coefficient_cash=+0.06564 | Q_ratio=+2.0004
  [CHI-FS] CASH fold 2: selected 50/62 features | top preview: ['cnt_fi_inst_appl_12m', 'cnt_lender_has2pay_last_24m', 'cnt_report_institution_12m', 'worst_credit_status', 'cnt_month_from_last_appl', 'last_offer_limit', 'cli_age', 'cnt_total_payment', 'cnt_month_status1', 'cnt_fail_tls_3m', 'cli_emp_period', 'avg_sign_vol']
  fold 2: Qini_coefficient_cash=+0.05559 | Q_ratio=+1.8462
  [CHI-FS] CASH fold 3: selected 48/62 features | top preview: ['cnt_fi_inst_appl_12m', 'cnt_report_institution_12m', 'cn

,model,Qini_coefficient_cash,Qini_coefficient_card,Qini_coefficient_total
0,INT_RF,0.058476,0.039081,0.097557



========== RUN DIRECT UPLIFT TREE/RF MODELS ==========

[CASH] CausalML UpliftRandomForest criterion=CIT
  fold 1: Qini_coefficient_cash=+0.07116 | Q_ratio=+2.0845
  fold 2: Qini_coefficient_cash=+0.04794 | Q_ratio=+1.7298


## 6. Summary, fold-mean 5 folds, ensemble

In [ ]:

def summarize_scores_all():
    models = sorted(set(scores_oof["cash"]).union(scores_oof["card"]))
    rows = []
    for model_name in models:
        cash_met = qini_metrics(Y_cash, w_cash, scores_oof["cash"][model_name]) if model_name in scores_oof["cash"] else None
        card_met = qini_metrics(Y_card, w_card, scores_oof["card"][model_name]) if model_name in scores_oof["card"] else None
        row = {
            "model": model_name,
            "Qini_coefficient_cash": cash_met["qini_norm"] if cash_met else np.nan,
            "Qini_coefficient_card": card_met["qini_norm"] if card_met else np.nan,
        }
        row["Qini_coefficient_total"] = row["Qini_coefficient_cash"] + row["Qini_coefficient_card"]
        rows.append(row)
    return pd.DataFrame(rows).sort_values("Qini_coefficient_total", ascending=False).reset_index(drop=True)


def make_fold_qini_table():
    rows = []
    models = sorted(set(scores_oof["cash"]).union(scores_oof["card"]))
    for model_name in models:
        for fold in range(1, N_SPLITS + 1):
            row = {"model": model_name, "fold": fold}
            idx = fold_ids["cash"] == fold
            if model_name in scores_oof["cash"]:
                row["Qini_coefficient_cash"] = qini_metrics(Y_cash[idx], w_cash[idx], scores_oof["cash"][model_name][idx])["qini_norm"]
            else:
                row["Qini_coefficient_cash"] = np.nan
            idx = fold_ids["card"] == fold
            if model_name in scores_oof["card"]:
                row["Qini_coefficient_card"] = qini_metrics(Y_card[idx], w_card[idx], scores_oof["card"][model_name][idx])["qini_norm"]
            else:
                row["Qini_coefficient_card"] = np.nan
            row["Qini_coefficient_total"] = row["Qini_coefficient_cash"] + row["Qini_coefficient_card"]
            rows.append(row)
    return pd.DataFrame(rows).sort_values(["model", "fold"]).reset_index(drop=True)


def make_fold_mean_table(fold_qini_df):
    return (
        fold_qini_df
        .groupby("model", as_index=False)
        .agg(
            n_folds=("fold", "nunique"),
            Mean_Qini_coefficient_cash=("Qini_coefficient_cash", "mean"),
            Mean_Qini_coefficient_card=("Qini_coefficient_card", "mean"),
            Mean_Qini_coefficient_total=("Qini_coefficient_total", "mean"),
        )
        .sort_values("Mean_Qini_coefficient_total", ascending=False)
        .reset_index(drop=True)
    )


# Summary trước ensemble.
summary_before_ensemble = summarize_scores_all()
print("Base model Qini coefficient summary — CASH / CARD / TOTAL")
display(summary_before_ensemble.round(6))

# Rank ensembles: Qini chỉ quan tâm ranking, nên ensemble theo rank thường ổn định hơn average xác suất thô.
base_models = summary_before_ensemble["model"].tolist()
if base_models:
    scores_oof["cash"]["ENS_ALL_RANK"] = rank_average([scores_oof["cash"][m] for m in base_models if m in scores_oof["cash"]])
    scores_oof["card"]["ENS_ALL_RANK"] = rank_average([scores_oof["card"][m] for m in base_models if m in scores_oof["card"]])
    if X_test is not None:
        scores_test["cash"]["ENS_ALL_RANK"] = rank_average([scores_test["cash"][m] for m in base_models if m in scores_test["cash"]])
        scores_test["card"]["ENS_ALL_RANK"] = rank_average([scores_test["card"][m] for m in base_models if m in scores_test["card"]])

    top_models = summary_before_ensemble.head(min(TOPK_ENSEMBLE, len(summary_before_ensemble)))["model"].tolist()
    scores_oof["cash"][f"ENS_TOP{len(top_models)}_RANK"] = rank_average([scores_oof["cash"][m] for m in top_models if m in scores_oof["cash"]])
    scores_oof["card"][f"ENS_TOP{len(top_models)}_RANK"] = rank_average([scores_oof["card"][m] for m in top_models if m in scores_oof["card"]])
    if X_test is not None:
        scores_test["cash"][f"ENS_TOP{len(top_models)}_RANK"] = rank_average([scores_test["cash"][m] for m in top_models if m in scores_test["cash"]])
        scores_test["card"][f"ENS_TOP{len(top_models)}_RANK"] = rank_average([scores_test["card"][m] for m in top_models if m in scores_test["card"]])

    # Weighted ensemble theo Qini total dương trên OOF.
    weights = summary_before_ensemble.set_index("model").loc[top_models, "Qini_coefficient_total"].clip(lower=0).to_numpy()
    if weights.sum() > 0:
        ens_name = f"ENS_TOP{len(top_models)}_WEIGHTED_RANK"
        scores_oof["cash"][ens_name] = rank_average([scores_oof["cash"][m] for m in top_models if m in scores_oof["cash"]], weights=weights)
        scores_oof["card"][ens_name] = rank_average([scores_oof["card"][m] for m in top_models if m in scores_oof["card"]], weights=weights)
        if X_test is not None:
            scores_test["cash"][ens_name] = rank_average([scores_test["cash"][m] for m in top_models if m in scores_test["cash"]], weights=weights)
            scores_test["card"][ens_name] = rank_average([scores_test["card"][m] for m in top_models if m in scores_test["card"]], weights=weights)

summary = summarize_scores_all()
print("Final Qini coefficient summary — CASH / CARD / TOTAL")
display(summary.round(6))

fold_qini_logs = make_fold_qini_table()
print("Fold-level Qini coefficient — CASH / CARD / TOTAL")
display(fold_qini_logs.round(6))

fold_mean_qini_by_model = make_fold_mean_table(fold_qini_logs)
print("Mean Qini coefficient over 5 folds — CASH / CARD / TOTAL")
display(fold_mean_qini_by_model.round(6))

best_model = summary.iloc[0]["model"]
print("Best model by OOF total Qini:", best_model)


## 7. Correlation and Qini curves

In [ ]:

def correlation_table(scores_dict):
    if not scores_dict:
        return pd.DataFrame()
    return pd.DataFrame(scores_dict).corr(method="spearman")

print("Cash score Spearman correlation:")
display(correlation_table(scores_oof["cash"]).round(3))
print("Card score Spearman correlation:")
display(correlation_table(scores_oof["card"]).round(3))


def plot_qini_compare(target_name, y, w_event, score_dict, top_k=6):
    local_rows = []
    for model_name, score in score_dict.items():
        m = qini_metrics(y, w_event, score)
        local_rows.append((model_name, m["qini_norm"]))
    keep_models = [m for m, _ in sorted(local_rows, key=lambda x: x[1], reverse=True)[:top_k]]

    plt.figure(figsize=(10, 6))
    xp, qp = qini_curve(y, w_event, oracle_score(y, w_event))
    ate = qp[-1]
    plt.plot([0, 1], [0, ate], "--", linewidth=1.2, label=f"Random baseline | ATE={ate:+.5f}")
    plt.plot(xp, qp, ":", linewidth=1.4, label="Perfect ranking")

    for model_name in keep_models:
        x, q = qini_curve(y, w_event, score_dict[model_name])
        m = qini_metrics(y, w_event, score_dict[model_name])
        plt.plot(x, q, linewidth=2, label=f"{model_name} | Qini_coeff={m['qini_norm']:+.4f}")

    plt.title(f"Qini Curve — {target_name.upper()}")
    plt.xlabel("Fraction of population sorted by predicted uplift")
    plt.ylabel("Qini")
    plt.grid(alpha=0.3)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

plot_qini_compare("cash", Y_cash, w_cash, scores_oof["cash"])
plot_qini_compare("card", Y_card, w_card, scores_oof["card"])


## 8. Export outputs

In [ ]:
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_path = OUTPUT_DIR / "qini_summary.csv"
summary.to_csv(summary_path, index=False)
print("Saved:", summary_path)

fold_path = OUTPUT_DIR / "fold_qini_logs.csv"
fold_qini_logs.to_csv(fold_path, index=False)
print("Saved:", fold_path)

fold_mean_path = OUTPUT_DIR / "fold_mean_qini_by_model.csv"
fold_mean_qini_by_model.to_csv(fold_mean_path, index=False)
print("Saved:", fold_mean_path)

run_log_df = pd.DataFrame(run_log)
run_log_path = OUTPUT_DIR / "run_log.csv"
run_log_df.to_csv(run_log_path, index=False)
print("Saved:", run_log_path)


# Save fold-safe Chi feature selection logs.
fs_log_df = pd.DataFrame(UPLIFT_FS_LOGS).drop_duplicates(subset=["target_runner", "fold"]) if globals().get("UPLIFT_FS_LOGS") else pd.DataFrame()
fs_log_path = OUTPUT_DIR / "uplift_chi_feature_selection_logs.csv"
fs_log_df.to_csv(fs_log_path, index=False)
print("Saved:", fs_log_path)

# Save OOF scores từng model.
oof_out = pd.DataFrame()
oof_out[ID_COL if ID_COL else "row_id"] = feat[ID_COL].values if ID_COL else np.arange(len(feat))
oof_out["treatment_cash"] = W
oof_out["cash_sign_30d"] = Y_cash
oof_out["card_sign_30d"] = Y_card
for name, score in scores_oof["cash"].items():
    oof_out[f"cash_{name}"] = score
for name, score in scores_oof["card"].items():
    oof_out[f"card_{name}"] = score

oof_path = OUTPUT_DIR / "oof_model_scores.csv"
oof_out.to_csv(oof_path, index=False)
print("Saved:", oof_path)

# Final train scores lấy theo best model by total Qini.
final_train = pd.DataFrame()
final_train[ID_COL if ID_COL else "row_id"] = feat[ID_COL].values if ID_COL else np.arange(len(feat))
final_train["best_model"] = best_model
final_train["uplift_cash_score"] = scores_oof["cash"][best_model]
final_train["uplift_card_score"] = scores_oof["card"][best_model]
final_train["cash_vs_card_score"] = final_train["uplift_cash_score"] - final_train["uplift_card_score"]
final_train["recommended_strategy"] = np.where(final_train["cash_vs_card_score"] >= 0, "Cash", "Card")

final_train_path = OUTPUT_DIR / "final_scores_train.csv"
final_train.to_csv(final_train_path, index=False)
print("Saved:", final_train_path)

if X_test is not None and best_model in scores_test["cash"] and best_model in scores_test["card"]:
    sub = pd.DataFrame()
    sub[test_id.name if test_id is not None else "row_id"] = test_id.values if test_id is not None else np.arange(len(X_test))
    sub["best_model"] = best_model
    sub["uplift_cash_score"] = scores_test["cash"][best_model]
    sub["uplift_card_score"] = scores_test["card"][best_model]
    sub["cash_vs_card_score"] = sub["uplift_cash_score"] - sub["uplift_card_score"]
    sub["recommended_strategy"] = np.where(sub["cash_vs_card_score"] >= 0, "Cash", "Card")
    sub_path = OUTPUT_DIR / "submission.csv"
    sub.to_csv(sub_path, index=False)
    print("Saved:", sub_path)
else:
    print("No test feature found or best_model has no test score -> submission.csv not created.")

config = {
    "SEED": SEED,
    "N_SPLITS": N_SPLITS,
    "N_JOBS": N_JOBS,
    "RUN_PRESET": RUN_PRESET,
    "REQUESTED_BASE_MODELS": REQUESTED_BASE_MODELS,
    "EXTRA_STRONG_BASE_MODELS": EXTRA_STRONG_BASE_MODELS,
    "RUN_MODEL_KINDS": RUN_MODEL_KINDS,
    "RUN_LEARNERS": RUN_LEARNERS,
    "RUN_OPTIONAL_CAUSALML_TREE": RUN_OPTIONAL_CAUSALML_TREE,
    "RUN_OPTIONAL_UTBOOST": RUN_OPTIONAL_UTBOOST,
    "DEEP_KAN_WIDTH_HIDDEN": DEEP_KAN_WIDTH_HIDDEN,
    "DEEP_KAN_GRID": DEEP_KAN_GRID,
    "DEEP_KAN_K": DEEP_KAN_K,
    "DEEP_KAN_STEPS": DEEP_KAN_STEPS,
    "DEEP_KAN_BATCH_SIZE": DEEP_KAN_BATCH_SIZE,
    "DEEP_KAN_LR": DEEP_KAN_LR,
    "DEEP_KAN_MAX_TRAIN_ROWS": DEEP_KAN_MAX_TRAIN_ROWS,
    "KNN_MAX_TRAIN_ROWS": KNN_MAX_TRAIN_ROWS,
    "KNN_NEIGHBORS": KNN_NEIGHBORS,
    "TOPK_ENSEMBLE": TOPK_ENSEMBLE,
    "UPLIFT_FS_ENABLED": UPLIFT_FS_ENABLED,
    "UPLIFT_FS_METHOD": UPLIFT_FS_METHOD,
    "UPLIFT_FS_TOPK_PER_TARGET": UPLIFT_FS_TOPK_PER_TARGET,
    "UPLIFT_FS_N_BINS": UPLIFT_FS_N_BINS,
    "UPLIFT_FS_MIN_BIN_SIZE": UPLIFT_FS_MIN_BIN_SIZE,
    "UPLIFT_FS_CHI_EPS": UPLIFT_FS_CHI_EPS,
    "UPLIFT_FS_USE_UNION_CASH_CARD": UPLIFT_FS_USE_UNION_CASH_CARD,
    "FULL_FEATURE_COUNT": len(FEAT_COLS),
    "FS_LOG_PATH": str(fs_log_path),
    "available_packages": {k: bool(v) for k, v in AVAILABLE.items()},
    "finished_models": list(scores_oof["cash"].keys()),
    "best_model": str(best_model),
}
config_path = OUTPUT_DIR / "run_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
print("Saved:", config_path)


## 9. Cách đọc kết quả — TOP2 + Fold-safe Chi filter

- `qini_summary.csv`: bảng tổng hợp Qini OOF của từng model được giữ lại, gồm `Qini_coefficient_cash`, `Qini_coefficient_card`, `Qini_coefficient_total`.
- `fold_qini_logs.csv`: Qini từng fold để kiểm tra độ ổn định.
- `fold_mean_qini_by_model.csv`: trung bình 5 fold của từng model.
- `uplift_chi_feature_selection_logs.csv`: feature được chọn bởi Chi filter trong từng fold.
- `oof_model_scores.csv`: OOF score của các model được giữ lại.
- `final_scores_train.csv`: score train theo best model tổng Qini.
- `submission.csv`: chỉ được tạo nếu notebook tìm thấy file test feature.

Notebook này đã cố định `TOP_MODEL_NAMES` gồm 2 model: `INT_RF` và `UPLIFT_RF_CIT`. Feature selection dùng Chi filter và chạy bên trong từng fold để hạn chế leakage từ validation fold.
